# **VTL Field Decomposition Notebook**
### *Gradient-Based Compositional Masking for Kernel Metrics (Δx, rᵥ, ρᵣ, μ, xₚ, θ, ds)*

This notebook implements the **field decomposition layer** of the Visual Thinking Lens (VTL).  
It converts an input image into a structured set of **gradient-derived fields, masks, and compositional primitives** that reveal the image’s spatial priors:

- void structure  
- mass concentration  
- gradient islands  
- attractor ridges  
- orientation flow  
- compositional spine  

These components form the computational substrate for the VTL kernel metrics:

- **Δx** — placement offset  
- **rᵥ** — void ratio  
- **ρᵣ** — packing density  
- **μ** — cohesion of structural islands  
- **xₚ** — peripheral-pull vector  
- **θ** — orientation stability  
- **ds** — structural thickness  

> **This notebook does not score images.**  
It builds the *structural ground truth* used by Sketcher, VCLI-G, and RIDP.

---

## **What This Notebook Provides**

This notebook extracts the **compositional primitives** of an image using gradient-field analysis:

1. **Void Mask** — low-pressure regions (rᵥ)  
2. **Mass Mask** — high-gradient structural mass (ρᵣ)  
3. **Edge / Boundary Mask** — peripheral pull behavior (xₚ)  
4. **Directional Mask** — orientation coherence (θ)  
5. **Cohesion Mask** — gradient islands for μ  
6. **Ridge / Attractor Mask** — Hessian-based attractor lines  
7. **Skeleton Mask** — structural spine for ds  

Collectively these define the **compositional field** — the raw spatial geometry the VTL reads.

---

## **Mask → Metric Mapping Table**

| **Mask Type**         | **Kernel Metric(s)** | **Field Interpretation** |
|-----------------------|----------------------|--------------------------|
| **Void Mask**         | rᵥ                   | Low-pressure regions / breathing space |
| **Mass Mask**         | ρᵣ                   | Packed structural mass / high-pressure zones |
| **Boundary Mask**     | xₚ                   | Peripheral compositional pull |
| **Directional Mask**  | θ                    | Orientation coherence and flow |
| **Cohesion Mask**     | μ                    | Structural islands → fragmentation vs unity |
| **Attractor Mask**    | — (supports xₚ, θ)   | Gradient ridges / structural attractors |
| **Skeleton Mask**     | ds                   | Structural thickness / volumetric vs filamentary |

---

## **How This Fits Into the VTL Architecture**

This notebook implements the **pre-metric layer**:

1. Load image  
2. Convert to perceptual luminance  
3. Build multi-scale Sobel gradient fields  
4. Derive compositional masks  
5. Convert masks → metric operations (Δx, rᵥ, ρᵣ, μ, xₚ, θ, ds)  
6. Render overlays and diagnostic views  

It defines the **raw geometry** on which all higher-level engines operate:
- **Sketcher Lens** — structural pressure  
- **VCLI-G** — perceptual load  
- **RIDP** — reverse decomposition  

---

# **Notebook Roadmap (Recommended Order)**

### **0 — Setup**
- Install dependencies  
- Upload interface  

---

### **1 — Perceptual Foundation**
1. **LAB Luminance vs Sobel Gradient Comparison**  
2. **Six-Panel Gradient-Field Breakdown**  
3. **Advanced Field Diagnostics**  
   - vectors, isolines, basins, ridges, torque wheel

---

### **2 — CV Primitives → VTL Primitives**
4. **CORE CV Techniques vs VCLI-G Primitives**  
5. **Four-Panel Gradient / Void / Skeleton Visualization**

---

### **3 — Integrated Composition**
6. **Full Compositional Overlay (Δx, xₚ, μ, void basins, spine)**

---

### **4 — Comparative Structure**
7. **Side-by-Side Compositional Comparison**  
8. **Compositional Fingerprint (Radar / Spider Chart)**  

---

### **5 — Intent & Strategy Indicators**
9. **Weighted Attention Heatmap**  
10. **Quadrant Weight Distribution (3×3 Grid)**  

---

### **6 — Metric Pipeline (Topology → Entropy → μ)**
11. **Cohesion Visual Companion (μ)**  
    - islands → entropy → cohesion  

---

## **Summary**

This notebook exposes the **mechanics of compositional intelligence**.  
It transforms an image into a set of spatial fields — pressure, void topology, flow coherence, mass distribution, attractor lines, and backbone structure.  

These reveal:
- the inductive biases of image models  
- the intentionality of artists  
- the hidden geometry beneath a composition  

This is the decomposition layer for the Visual Thinking Lens —  
the part that treats an image not as a picture, but as a **field of forces**.

---
---
---

## Optional Pre-Processing: Anisotropic Diffusion (Perona–Malik)

### Why this matters

The VTL pipeline operates on **Sobel gradient fields**, which are highly sensitive to:

- **JPEG artifacts / film grain / sensor noise**
- **Over-compressed web images**
- **Textured substrates** (canvas, paper, fabric)

Without any pre-processing, high-frequency noise can be misread as:

- extra “structural mass” → inflating **ρᵣ** (packing density)  
- many tiny gradient islands → fragmenting **μ** (cohesion)  
- false ridges / edges → perturbing **xₚ**, **ds**, and island counts  

Percentile thresholds (e.g. top 25% for mass) help make the system exposure-invariant,  
but they still assume the **gradient histogram is dominated by structure, not noise**.

### Role of anisotropic diffusion

To mitigate this, the notebook can optionally apply an **edge-preserving smoothing step** before constructing Sobel gradients:

> **Perona–Malik anisotropic diffusion**  
> non-linear, gradient-aware diffusion that smooths low-contrast noise  
> while preserving high-contrast structural edges.

Conceptually, the pipeline becomes:

```text
Original Image
→ (optional) Anisotropic Diffusion
→ Sobel Gradient Field G′
→ VTL Masks (void, mass, islands, ridges, skeleton)
→ Kernel Metrics (Δx, rᵥ, ρᵣ, μ, xₚ, θ, ds)

In [ ]:
USE_DIFFUSION = False   # change to True to globally enable

if USE_DIFFUSION:
    gray_for_gradient = anisotropic_diffusion(gray, niter=10, kappa=20, gamma=0.1)
else:
    gray_for_gradient = gray

## When to Enable Anisotropic Diffusion (Perona–Malik)

Diffusion is **optional**. It does not change what the kernel measures, only how clean the gradient field is before VTL reads it.

Use it when the image has **accidental noise** that would lie to the gradient field.  
Avoid it when the “noise” is actually **intentional texture** or markmaking.

---

### ✅ Good Candidates for Diffusion

These are cases where anisotropic diffusion usually *helps* VTL:

#### 1. Compression-Heavy / JPEG Images
- Web-saved JPGs, screenshots, recompressed images  
- Midjourney / SDXL / GPT outputs that show faint blocky or tiled artifacts  

**Why:**  
Compression edges and ringing appear as fake high-gradients.  
They inflate **ρᵣ** (packing density), fragment **μ** (cohesion), and contaminate voids.

---

#### 2. Grainy Film Scans or High-ISO Photos
- Scanned negatives / prints  
- Low-light high-ISO photos (mobile or DSLR)  

**Why:**  
Film grain and sensor noise get treated as structural mass.  
Gradient islands explode in count, μ collapses, and voids crumble.

---

#### 3. AI Images with Procedural Surface Noise
- Texture-heavy generative outputs  
- “Over-detailed” surfaces with micro-speckle  

**Why:**  
Procedural micro-noise creates a fake crust of gradients.  
Void masks and island maps stop reflecting actual form.

---

#### 4. Over-Sharpened / Tone-Mapped Images
- Strong “clarity” / sharpening / HDR filters  
- Images with haloed edges  

**Why:**  
Sharpening boosts all edges, including unimportant ones.  
ρᵣ spikes, and gradient fields over-report structure everywhere.

---

#### 5. Images with Small Text / UI / Watermarks
- Screenshots of interfaces  
- Photos of screens with UI chrome  
- Watermarked images  

**Why:**  
Tiny text and UI elements become dense edge clusters.  
They dominate island counting and distort cohesion.

---

### ❌ Cases Where Diffusion Should Usually Stay OFF

Here, the “noise” is actually part of the **intended** structure:

#### 1. Charcoal / Graphite / Pastel / Dry Media
- Grain and micro-variation are part of pressure and form.  
- Smoothing erases the evidence of markmaking.

#### 2. Painterly Surfaces
- Visible brushwork, impasto, broken color, scumbling.  
- These textures carry structural intent, not just detail.

#### 3. Intentionally Rough / Noisy Images
- Goya-like roughness, Bacon, Freud, expressive abstraction.  
- The unevenness is the point.

#### 4. Fine Architectural / Geometric Detail
- Building facades with small windows, grills, lattice, tiling.  
- Diffusion can wipe out crucial micro-geometry that defines rhythm.

---

### Simple Rule of Thumb

> **Turn diffusion ON when the noise is accidental.**  
> **Turn diffusion OFF when the texture is intentional.**

Ask one question:

> “If I were critiquing this as an artwork,  
> would I talk about this texture/noise as *part of the piece*  
> or as *a technical flaw*?”

- If it’s a *flaw* → diffusion **ON** is reasonable.  
- If it’s *part of the work* → keep diffusion **OFF**.

---

### Practical Use in This Notebook

- Control is via a single flag:

```python
USE_DIFFUSION = False  # set to True to enable Perona–Malik pre-pass

In [ ]:
# ================================================
# OPTIONAL PRE-PROCESSING:
# Anisotropic Diffusion vs. Raw Sobel Gradients
# Shows how diffusion changes the gradient field
# ================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import sobel
from skimage.measure import label
from google.colab import files

# ---- Simple Perona–Malik anisotropic diffusion ----
def anisotropic_diffusion(img, niter=10, kappa=20, gamma=0.1):
    """
    Perona–Malik anisotropic diffusion (edge-preserving smoothing).
    img   : 2D float32 array in [0,1]
    niter : number of iterations
    kappa : conduction coefficient (larger = more smoothing)
    gamma : timestep (0 < gamma <= 0.25 for stability)
    """
    img = img.astype(np.float32)
    diff = img.copy()
    for _ in range(niter):
        # Neighbor differences
        north = np.zeros_like(diff)
        south = np.zeros_like(diff)
        east  = np.zeros_like(diff)
        west  = np.zeros_like(diff)

        north[1:, :] = diff[1:, :] - diff[:-1, :]
        south[:-1, :] = diff[:-1, :] - diff[1:, :]
        east[:, :-1]  = diff[:, :-1] - diff[:, 1:]
        west[:, 1:]   = diff[:, 1:] - diff[:, :-1]

        # Conduction coefficients (option 1: exponential)
        cN = np.exp(-(north / kappa) ** 2)
        cS = np.exp(-(south / kappa) ** 2)
        cE = np.exp(-(east  / kappa) ** 2)
        cW = np.exp(-(west  / kappa) ** 2)

        # Update
        diff += gamma * (cN * north + cS * south + cE * east + cW * west)

    diff = np.clip(diff, 0.0, 1.0)
    return diff

# ---- Simple island / cohesion proxy for demonstration ----
def cohesion_stats(grad, percentile=75):
    """
    Quick proxy to show impact on structure:
    - Threshold gradient at given percentile
    - Count connected islands
    - Compute simple entropy-based μ-like cohesion value
    """
    thr = np.percentile(grad, percentile)
    mask = grad > thr

    lbl, num = label(mask, return_num=True)
    if num == 0:
        return 0, 0.0

    sizes = np.bincount(lbl.ravel())[1:]  # skip background
    p = sizes / sizes.sum()
    H = -np.sum(p * np.log(p + 1e-12))
    H_max = np.log(len(sizes) + 1e-12)
    mu = 1.0 - (H / H_max)
    return int(num), float(mu)

# ---- Upload image ----
print("Upload an image to compare raw vs. diffused gradients...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + preprocess ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype("float32") / 255.0

# ---- Build gradients: raw vs diffusion ----
grad_raw = sobel(gray)

# You can tune these parameters:
USE_DIFFUSION = True
NITER = 10
KAPPA = 20
GAMMA = 0.1

if USE_DIFFUSION:
    gray_diff = anisotropic_diffusion(gray, niter=NITER, kappa=KAPPA, gamma=GAMMA)
else:
    gray_diff = gray.copy()

grad_diff = sobel(gray_diff)

# ---- Compute simple stats ----
islands_raw, mu_raw = cohesion_stats(grad_raw, percentile=75)
islands_diff, mu_diff = cohesion_stats(grad_diff, percentile=75)

print("\n=== Structural stats (demo only, not full VTL μ) ===")
print(f"Raw gradients   : islands = {islands_raw:4d}, cohesion-like μ ≈ {mu_raw:.3f}")
print(f"Diffused grads  : islands = {islands_diff:4d}, cohesion-like μ ≈ {mu_diff:.3f}")

# ---- Visual comparison ----
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original RGB")
axes[0, 0].axis("off")

axes[0, 1].imshow(gray_diff, cmap="gray")
axes[0, 1].set_title("After Anisotropic Diffusion")
axes[0, 1].axis("off")

im1 = axes[1, 0].imshow(grad_raw, cmap="magma")
axes[1, 0].set_title("|∇I| Raw Sobel")
axes[1, 0].axis("off")
fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04)

im2 = axes[1, 1].imshow(grad_diff, cmap="magma")
axes[1, 1].set_title("|∇I′| After Diffusion")
axes[1, 1].axis("off")
fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

---
---
---

# 1. LAB Luminance vs. Sobel Gradient Comparison  
### Showing what VCLI-G *actually* measures — luminance gradients, not RGB

This diagnostic clarifies the perceptual foundation of the entire kernel.  
It places three views side-by-side:

1. **Original RGB**  
   The raw image, as seen by the user.

2. **LAB Luminance (L*)**  
   A perceptually uniform measure of brightness — much closer to how humans actually see luminance.  
   LAB-L removes hue and saturation and isolates the true brightness structure.

3. **Sobel Gradient Magnitude**  
   The “pressure field” used by VCLI-G, Δx, μ, ρᵣ, rᵥ, and xₚ.  
   This reveals brightness *change*, not brightness *value*.

---

## **Why this comparison matters**
VCLI-G is not an RGB analyzer.  
It is a **luminance-gradient analyzer** — and this cell proves that.

- LAB-L and grayscale have **>0.95 correlation** for most natural images.  
- Sobel magnitude highlights *where* brightness changes sharply — the true structural mass of the image.  
- Most compositional intelligence lives in the **derivatives**, not the pixel values.

---

## **What the cell provides**
- A self-contained upload flow  
- Three synchronized panels  
- Gradient statistics (mean, std, max)  
- A correlation score between LAB-L and grayscale  
- A brief textual interpretation  
- Consistent styling with your other diagnostic cells (“magma” colormap)

---

## **Interpretation**
This comparison demonstrates:

- Why Δx is computed from gradient mass rather than RGB centroids  
- Why μ (cohesion) emerges from gradient continuity rather than object labels  
- Why rᵥ (void ratio) aligns with low-gradient plateaus  
- Why xₚ (peripheral pull) depends on directional imbalance in luminance transitions  
- Why θ (orientation stability) derives from gradient direction, not subject layout  

In short:  
**VCLI-G sees structure the same way human perception does — through changes in light, not color.**


In [ ]:
# ============================================================
# LAB LUMINANCE vs. SOBEL GRADIENT COMPARISON
# Shows what VCLI-G actually "sees" - luminance gradients, not RGB
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure
from google.colab import files

# ---- Upload image ----
print("Upload an image to analyze LAB luminance vs. gradient field...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + convert to multiple color spaces ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

# Convert to LAB color space
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
L_channel = img_lab[:, :, 0]  # Luminance channel (0-255)
L_normalized = L_channel.astype("float32") / 255.0

# ---- Compute Sobel gradient magnitude ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

# ---- Compute statistics for comparison ----
# Correlation between LAB-L and grayscale
corr = np.corrcoef(L_normalized.flatten(), gray_f.flatten())[0, 1]

# Gradient statistics
grad_mean = grad_mag_norm.mean()
grad_std = grad_mag_norm.std()
grad_max = grad_mag_norm.max()

# ---- PLOT 3-PANEL COMPARISON ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Original RGB
axes[0].imshow(img_rgb)
axes[0].set_title("Original (RGB)", fontsize=14, weight='bold')
axes[0].axis("off")

# 2. LAB Luminance Channel
axes[1].imshow(L_channel, cmap='gray')
axes[1].set_title("LAB L* Channel\n(Perceptual Luminance)", fontsize=14, weight='bold')
axes[1].axis("off")

# Add text annotation
axes[1].text(10, 30, f"LAB-L ≈ Grayscale\nr = {corr:.3f}",
            color='yellow', fontsize=11, weight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# 3. Sobel Gradient Magnitude (what VCLI-G actually measures)
axes[2].imshow(grad_mag_norm, cmap='magma')
axes[2].set_title("Sobel Gradient Magnitude\n(Rate of Luminance Change)",
                 fontsize=14, weight='bold')
axes[2].axis("off")

# Add gradient statistics
axes[2].text(10, 30,
            f"Mean: {grad_mean:.3f}\n"
            f"Std: {grad_std:.3f}\n"
            f"Max: {grad_max:.3f}",
            color='white', fontsize=11, weight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

plt.suptitle("What VCLI-G Sees: Luminance → Gradients → Geometric Primitives",
             fontsize=16, weight='bold', y=1.02)

plt.tight_layout()
plt.show()

# ---- Print interpretation ----
print("\n" + "="*60)
print("INTERPRETATION:")
print("="*60)
print(f"• LAB-L correlation with grayscale: r = {corr:.3f}")
print("  (High correlation confirms LAB-L ≈ perceptual brightness)")
print()
print("• VCLI-G measures GRADIENTS (rate of change), not raw brightness")
print("  - High gradient = edges, boundaries, tonal transitions")
print("  - Low gradient = smooth areas, voids, breathing zones")
print()
print(f"• Gradient field statistics:")
print(f"  - Mean magnitude: {grad_mean:.3f} (overall edge density)")
print(f"  - Std deviation: {grad_std:.3f} (variation in edge strength)")
print(f"  - Maximum: {grad_max:.3f} (sharpest transitions)")
print()
print("This is the 'pressure field' that drives all VCLI-G measurements:")
print("  - Centroid wander tracks gradient-weighted mass")
print("  - Void detection finds low-gradient regions")
print("  - Curvature analyzes gradient flow direction")
print("  - Occlusion uses orientation entropy of gradients")
print("="*60)

# 2. Six-Panel Advanced Gradient-Field Breakdown — Documentation

These six panels expose the full internal structure of the gradient field: magnitude, direction, attraction, fragmentation, and topological flow.  
Together they reveal the operational geometry behind Δx, rᵥ, ρᵣ, μ, xₚ, θ, and ds.

---

## **1. Gradient Magnitude (Pressure Map)**
**What this is:**  
A Sobel-derived pressure map showing how strongly luminance changes at every pixel.

**Reveals:**  
- Structural mass  
- Edge pressure  
- Dominant carriers of Δx  
- Primary contributors to cohesion μ  
- Pack density ρᵣ in high-gradient regions

**Kernel tie-ins:**  
- **Δx** from gradient centroid  
- **ρᵣ** from concentration of bright regions  
- **μ** from cluster continuity  
- **xₚ** from left–right imbalance in pressure  
- **ds** from thickness of concentrated mass

---

## **2. Raw Sobel X / Sobel Y (Directional Slopes)**
**What this is:**  
The horizontal (Gx) and vertical (Gy) components of the gradient.  
These show *how* brightness changes, not just where.

**Reveals:**  
- Left/right vs up/down rate of change  
- Torque seeds  
- Orientation asymmetries  
- Basis of θ (orientation stability)

**Kernel tie-ins:**  
- **θ** from dominant slope orientation  
- **xₚ** from directional imbalance  
- **μ** from consistent directional clusters

---

## **3. Directional Vector Field (Quiver Map)**
**What this is:**  
A dense orientation field showing the local direction of change at each sampled point.

**Reveals:**  
- Flow direction  
- Rotational drift  
- Local torque  
- How the composition “breathes” directionally

**Kernel tie-ins:**  
- **θ** formalized  
- **μ** visible as coherent directional flow  
- **xₚ** manifests as global field bias  
- **ρᵣ** indirectly through vector clustering

---

## **4. Gradient Basin Contours (Isolines)**
**What this is:**  
Topographic contours of gradient magnitude.  
Shows pressure basins and ridges like a geographic map.

**Reveals:**  
- Attractor basins  
- Forbidden void zones  
- Local energy wells  
- Structural ridgelines (precursors to skeleton)

**Kernel tie-ins:**  
- **Δx** from basin center-of-mass  
- **xₚ** from asymmetry of gradient basins  
- **μ** from how basins merge or fragment  
- **rᵥ** from empty plateau regions

---

## **5. Gradient Island Clustering (Cohesion Field μ)**
**What this is:**  
Connected-component segmentation of high-gradient zones.  
Each island is a coherent structural fragment.

**Reveals:**  
- Cohesion vs fragmentation  
- Discrete structural “objects”  
- Mass articulation  
- The limits of μ

**Kernel tie-ins:**  
- **μ** directly visualized  
- **ρᵣ** from island density  
- **ds** from island thickness  
- **xₚ** from cluster asymmetry

---

## **6. Hessian Ridge Detection (Attractor Lines)**
**What this is:**  
A Hessian-eigenvalue ridge detector tracing lines of maximal structural continuity.  
This becomes the *structural attractor map*.

**Reveals:**  
- Backbone flow  
- Tension pathways  
- Structural attractors  
- Fracture and collapse lines  
- The internal “spine” of the image

**Kernel tie-ins:**  
- **μ** from ridge strength and continuity  
- **θ** from ridge orientation patterns  
- **ds** from ridge thickness  
- **xₚ** often aligns with dominant attractor direction


In [ ]:
# ============================================================
# UPLOAD + 6-PANEL GRADIENT / VOID / SKELETON / SOBEL / QUIVER
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from skimage import exposure
from google.colab import files

# ---- Upload image ----
print("Upload an image...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + preprocess ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

# ---- Sobel gradients ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)

# Gradient magnitude (pressure field)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

# ---- Void mask ----
void_thresh = np.percentile(grad_mag, 60)
void_mask = grad_mag < void_thresh

# ---- Skeleton ----
skel = skeletonize(grad_mag_norm > np.percentile(grad_mag_norm, 85))

# ---- Downsampled gradient vectors for quiver ----
step = max(min(H, W) // 40, 5)  # adaptive step so it isn't insane
ys, xs = np.mgrid[0:H:step, 0:W:step]
gx_s = gx[0:H:step, 0:W:step]
gy_s = gy[0:H:step, 0:W:step]

# Normalize arrows for display (avoid huge vectors)
mag_s = np.sqrt(gx_s**2 + gy_s**2) + 1e-6
u = gx_s / mag_s
v = -gy_s / mag_s  # flip y for image coords

# ---- PLOT 6-PANEL FIGURE ----
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
ax = axes.ravel()

# 1. Original
ax[0].imshow(img)
ax[0].set_title("Original Image")
ax[0].axis("off")

# 2. Gradient magnitude + quiver
ax[1].imshow(img)
ax[1].imshow(grad_mag_norm, cmap="magma", alpha=0.6)
ax[1].quiver(xs, ys, u, v, color="cyan", scale=30, width=0.002, alpha=0.7)
ax[1].set_title("Gradient Magnitude + Vectors")
ax[1].axis("off")

# 3. Void mask overlay
void_overlay = np.zeros((*void_mask.shape, 4))
void_overlay[..., 2] = 1.0
void_overlay[..., 3] = void_mask.astype(float) * 0.35
ax[2].imshow(img)
ax[2].imshow(void_overlay)
ax[2].set_title("Void Mask (Breathing Zones)")
ax[2].axis("off")

# 4. Skeleton overlay
skel_overlay = np.zeros((*skel.shape, 4))
skel_overlay[..., :3] = 1.0
skel_overlay[..., 3] = skel.astype(float) * 0.9
ax[3].imshow(img)
ax[3].imshow(skel_overlay)
ax[3].set_title("Gradient Skeleton (Structural Spine)")
ax[3].axis("off")

# 5. Raw Sobel X
ax[4].imshow(gx, cmap="gray")
ax[4].set_title("Sobel X (Horizontal Change)")
ax[4].axis("off")

# 6. Raw Sobel Y
ax[5].imshow(gy, cmap="gray")
ax[5].set_title("Sobel Y (Vertical Change)")
ax[5].axis("off")

plt.tight_layout()
plt.show()

# 3. Advanced Field Diagnostics
### Quiver • Torque Wheel • Basins • Islands • Ridge Attractors

This suite exposes the deeper mechanics of the gradient field — not just where structure exists,  
but **how it flows, rotates, fragments, stabilizes, and collapses**.  
These diagnostics reveal the operational geometry behind Δx, rᵥ, ρᵣ, μ, xₚ, θ, and ds with maximal interpretability.

---

## **1. Directional Quiver Map (Local Flow Field)**
**What this is:**  
A sparse sampling of gradient orientation vectors overlaid onto the original image.

**Reveals:**  
- Local directional flow  
- Hidden torque pockets  
- Boundary interactions  
- Orientation coherence vs turbulence

**Interpretation:**  
This is the closest visualization to “how the image wants to move.”  
Flow alignment → high μ (cohesion).  
Flow turbulence → fracture potential.

**Kernel links:**  
- **θ**: orientation stability from directional spread  
- **μ**: smooth vs broken vector continuity  
- **xₚ**: global directional drift  
- **rᵥ**: voids show no directional force

---

## **2. Torque Wheel Indicator (Signed Rotational Bias)**
**What this is:**  
A circular visualization of signed orientation, mapping gradient direction to torque (clockwise vs counterclockwise).

**Reveals:**  
- Global rotation bias  
- Structural swirl or counter-swirl  
- The “handedness” of the composition  
- Stress points where orientation abruptly flips

**Interpretation:**  
The torque wheel shows whether a composition is:
- pulling left,  
- pulling right,  
- lifting upward,  
- collapsing downward,  
- or spiraling.

**Kernel links:**  
- **θ**: dominant orientation angle  
- **μ**: torque consistency  
- **xₚ**: torque asymmetry amplifies pull  
- **ds**: broader torque bandwidth = thicker structure

---

## **3. Gradient Basin Contours (Isoline Topography)**
**What this is:**  
Contour lines of equal gradient magnitude — a topographic map of structural pressure.

**Reveals:**  
- Pressure basins  
- Forbidden plateaus  
- Ridge boundaries  
- Transition zones between mass and void

**Interpretation:**  
Isoline topology shows where the image *settles*.  
Basins = attractor wells.  
Ridges = tension boundaries.

**Kernel links:**  
- **Δx**: basin center-of-mass  
- **xₚ**: basin asymmetry drives pull  
- **μ**: merged basins → higher cohesion  
- **rᵥ**: basins framed by void plateaus

---

## **4. Gradient Island Clustering (Cohesion μ Map)**
**What this is:**  
Connected-component segmentation of high-gradient regions, each shown as a distinct island.

**Reveals:**  
- Structural fragmentation vs unity  
- Presence of multiple competing masses  
- How tightly the composition “hangs together”  
- μ as discrete geometry

**Interpretation:**  
One large island → unified composition.  
Many small islands → fractured, noisy, low-cohesion structure.

**Kernel links:**  
- **μ**: direct visualization  
- **ρᵣ**: high island density = tight packing  
- **ds**: large island thickness = stronger mass  
- **xₚ**: asymmetric island distribution = directional pull

---

## **5. Hessian Ridge Attractors (Structural Spine Analysis)**
**What this is:**  
Ridge tracing using Hessian eigenvalue fields, revealing the underlying attractor network.

**Reveals:**  
- Structural backbone  
- Force-routing pathways  
- Compositional attractors  
- Stress fractures and collapse lines  
- Primary “reading routes” of the image

**Interpretation:**  
This is the internal nervous system of the image:  
Where force travels, where it stops, where it snaps.

**Kernel links:**  
- **μ**: ridge continuity = structural cohesion  
- **θ**: ridge direction = orientation stability  
- **ds**: ridge strength = mass thickness  
- **xₚ**: attractor direction predicts pull  
- **ρᵣ**: dense ridge networks imply concentrated structural load

---

## **Summary of Advanced Diagnostics**
These five diagnostics transform the gradient field from a simple edge map into a **dynamic system** with:  
- flow (quiver)  
- spin (torque wheel)  
- landscape (basins)  
- grouping (islands)  
- backbone (ridges)

Together, they expose **the full spatial prior machinery** operating beneath any image — human or AI-generated.


In [ ]:
# ============================================================
# ADVANCED FIELD DIAGNOSTICS
# Quiver, Torque Wheel, Basins, Islands, Ridge Attractors
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from skimage import exposure
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from skimage.measure import label, regionprops
from google.colab import files

# ---- Upload image ----
print("Upload an image...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + preprocess ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

# ---- Sobel gradients ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)

# Gradient magnitude (pressure field)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

# ---- Direction field (angles) ----
angles = np.arctan2(gy, gx)  # [-pi, pi]

# ---- Void mask ----
void_thresh = np.percentile(grad_mag, 60)
void_mask = grad_mag < void_thresh

# ---- Skeleton (for reference if needed) ----
skel = skeletonize(grad_mag_norm > np.percentile(grad_mag_norm, 85))

# ---- Downsampled gradient vectors for quiver ----
step = max(min(H, W) // 40, 5)  # adaptive step so it isn't too dense
ys, xs = np.mgrid[0:H:step, 0:W:step]
gx_s = gx[0:H:step, 0:W:step]
gy_s = gy[0:H:step, 0:W:step]

mag_s = np.sqrt(gx_s**2 + gy_s**2) + 1e-6
u = gx_s / mag_s
v = -gy_s / mag_s  # flip y for display coordinates

# ---- Isoline contours: we’ll just use grad_mag_norm ----
# (drawn later over either gray or original)

# ---- Cluster-colored gradient islands (for μ, packing, cohesion) ----
high_mask = grad_mag > np.percentile(grad_mag, 75)
lbl = label(high_mask)
n_labels = lbl.max()

cluster_rgb = np.zeros((*gray.shape, 3), dtype="float32")
if n_labels > 0:
    rng = np.random.default_rng(42)
    colors = rng.random((n_labels + 1, 3))  # random color per cluster
    for y in range(H):
        for x in range(W):
            lab_id = lbl[y, x]
            if lab_id > 0:
                cluster_rgb[y, x] = colors[lab_id]
else:
    cluster_rgb[:] = 0

# ---- Ridge detection using Hessian eigenvalues (attractors) ----
H_elems = hessian_matrix(gray_f, sigma=2.0, order='rc')
l1, l2 = hessian_matrix_eigvals(H_elems)  # pass tuple from hessian_matrix

ridge_strength = np.abs(l2)  # pick the stronger eigenvalue
ridge_thresh = np.percentile(ridge_strength, 90)
ridge_mask = ridge_strength > ridge_thresh

ridge_overlay = np.zeros((*gray.shape, 4), dtype="float32")
ridge_overlay[..., 0] = 1.0  # red
ridge_overlay[..., 3] = ridge_mask.astype(float) * 0.85

# ---- Torque wheel (orientation distribution, weighted by grad) ----
weights = grad_mag.flatten()
theta = angles.flatten()  # [-pi, pi]

num_bins = 36
bin_edges = np.linspace(-np.pi, np.pi, num_bins + 1)
hist, _ = np.histogram(theta, bins=bin_edges, weights=weights)

# Convert bin centers to [0, 2pi] for polar plot
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
theta_polar = (bin_centers + 2 * np.pi) % (2 * np.pi)

hist_norm = hist / (hist.max() + 1e-9)

# Signed rotational bias: >0 = CCW-dominant, <0 = CW-dominant (roughly)
bias = np.sum(weights * np.sin(theta)) / (np.sum(weights) + 1e-9)

# ---- Orientation histogram (linear) ----
ori_hist, ori_edges = np.histogram(theta, bins=num_bins, weights=weights)
ori_centers = 0.5 * (ori_edges[:-1] + ori_edges[1:])

# ============================================================
# PLOT 3x3 GRID
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
ax = axes.ravel()

# 1. Original
ax[0].imshow(img)
ax[0].set_title("Original")
ax[0].axis("off")

# 2. Gradient magnitude (pressure field)
ax[1].imshow(grad_mag_norm, cmap="gray")
ax[1].set_title("Gradient Magnitude (Pressure Field)")
ax[1].axis("off")

# 3. Quiver: orientation field over grayscale
ax[2].imshow(gray, cmap="gray")
ax[2].quiver(xs, ys, u, v, color="cyan", scale=30, width=0.002, alpha=0.7)
ax[2].set_title("Directional Quiver Map (Orientation Field)")
ax[2].axis("off")

# 4. Isoline contours: basins over gradient magnitude
ax[3].imshow(grad_mag_norm, cmap="gray")
ax[3].contour(grad_mag_norm, levels=10, linewidths=0.7)
ax[3].set_title("Isoline Contours (Gradient Basins)")
ax[3].axis("off")

# 5. Cluster-colored gradient islands
ax[4].imshow(cluster_rgb)
ax[4].set_title("Cluster Coloring of Gradient Islands")
ax[4].axis("off")

# 6. Ridge detection via Hessian eigenvalues
ax[5].imshow(img)
ax[5].imshow(ridge_overlay)
ax[5].set_title("Ridge Detection (Hessian Attractors)")
ax[5].axis("off")

# 7. Torque wheel (polar)
ax7 = plt.subplot(3, 3, 7, projection="polar")
ax7.bar(theta_polar, hist_norm, width=2*np.pi/num_bins, bottom=0.0, alpha=0.8)
ax7.set_title("Torque Wheel (Orientation Distribution)")

# 8. Orientation histogram (linear)
ax[7].plot(ori_centers, ori_hist)
ax[7].set_title("Orientation Histogram (Weighted by Gradient)")
ax[7].set_xlabel("Angle (radians)")
ax[7].set_ylabel("Weighted Count")

# 9. Bias readout + legend panel
ax[8].axis("off")
ax[8].text(
    0.05, 0.8,
    "Torque / Orientation Notes",
    fontsize=14,
    weight="bold",
)
ax[8].text(
    0.05, 0.6,
    f"Signed rotational bias ≈ {bias:.4f}\n"
    " > 0 → CCW-dominant\n"
    " < 0 → CW-dominant",
    fontsize=12,
)
ax[8].text(
    0.05, 0.35,
    "Interpretation:\n"
    " • Torque wheel: where edges lean / turn\n"
    " • Basins: stable pressure wells\n"
    " • Islands: cohesion / μ evidence\n"
    " • Ridges: attractor lines (Hessian)\n",
    fontsize=11,
)

plt.tight_layout()
plt.show()

# 4. CORE CV Techniques vs. VCLI-G Primitives  
### Otsu • Canny • Saliency • Distance Transform  
A 2×2 comparison revealing how classical CV methods map onto VCLI-G’s field primitives.

This diagnostic grid shows how **traditional computer vision operations** relate to the compositional metrics used in VCLI-G.  
Each panel exposes a classical technique and its conceptual correspondence to Δx, rᵥ, ρᵣ, μ, θ, and ds.

---

## **1. Otsu Thresholding → G₂ (Void Topology)**  
**What this is:**  
A global, histogram-based thresholding method that separates figure from ground without supervision.

**Reveals:**  
- Basic void topology  
- Foreground/Background partition  
- Coarse segmentation of structural vs empty space

**Why it matters to VCLI-G:**  
- Otsu approximates how **rᵥ (void ratio)** behaves under binary simplification.  
- Foreground ratio maps to **packing density (ρᵣ)**.  
- Coarse void mass corresponds to regions VCLI-G labels as “breathing zones.”

---

## **2. Canny Edge Detection → G₃ (Curvature Torque)**  
**What this is:**  
A classic multi-stage edge detector producing thin, clean structural boundaries.

**Reveals:**  
- Edge continuity  
- Directional curvature  
- Topological flow lines  
- Early-stage structural skeleton

**Why it matters to VCLI-G:**  
- Canny outlines **curvature torque** (a precursor to θ).  
- Edge density correlates with **packing density (ρᵣ)**.  
- These edges often evolve into the **ridge maps** seen in your Hessian attractor visualizations.

---

## **3. Spectral Residual Saliency Map → G₁ (Centroid Wander)**  
**What this is:**  
A frequency-domain saliency estimator highlighting “attention hotspots.”

**Reveals:**  
- Probable fixation points  
- High-information zones  
- Long-range spatial imbalance

**Why it matters to VCLI-G:**  
- The saliency centroid tends to drift similarly to **Δx** in your gradient-centroid calculations.  
- The saliency heatmap approximates **structural mass influence** even when gradients are low.  
- Saliency imbalance exposes the same **pull vectors** that xₚ formalizes.

---

## **4. Distance Transform → G₂ (Void Depth / Basin Penetration)**  
**What this is:**  
A transform computing the distance from every pixel to the nearest edge or boundary.

**Reveals:**  
- Depth inside voids  
- How far “into nothing” the frame can move  
- Shape of negative space  
- Convexity of void regions

**Why it matters to VCLI-G:**  
- Distance peaks represent **maximum void penetration**, a structural measure of rᵥ richness.  
- Distance gradients approximate **void basin slopes**, connecting directly to your isoline/basin views.  
- Void thickness influences the field’s **stability under torque** and the shape of Δx paths.

---

# Summary  
This 2×2 grid demonstrates that:

- **Otsu** approximates VCLI-G’s void topology (rᵥ).  
- **Canny** outlines curvature and torque precursors (θ, μ, ds).  
- **Saliency** predicts centroid drift (Δx) and structural pull (xₚ).  
- **Distance Transform** quantifies void depth (rᵥ, basin shape) and negative space complexity.

In other words:  
**Traditional CV already hints at VCLI-G’s primitives — but VCLI-G fuses them into a unified spatial prior framework.**


In [ ]:
# ============================================================
# CORE CV TECHNIQUES vs. VCLI-G PRIMITIVES
# Otsu, Canny, Saliency, Distance Transform
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import sobel
from scipy.ndimage import distance_transform_edt
from google.colab import files

# ---- Upload image ----
print("Upload an image to compare CV techniques with VCLI-G primitives...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + preprocess ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

# ============================================================
# 1. OTSU THRESHOLDING (Figure/Ground Separation)
# ============================================================
# Relates to: G2 (Void Topology) - automatic figure/ground segmentation
_, otsu_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
otsu_threshold = _

# Count foreground vs background pixels
fg_pixels = np.sum(otsu_mask > 0)
bg_pixels = np.sum(otsu_mask == 0)
fg_ratio = fg_pixels / (fg_pixels + bg_pixels)

# ============================================================
# 2. CANNY EDGE DETECTION (Structural Boundaries)
# ============================================================
# Relates to: G3 (Curvature) - clean edge contours for geometric analysis
med_val = np.median(gray)
lower = int(max(0, 0.66 * med_val))
upper = int(min(255, 1.33 * med_val))
canny_edges = cv2.Canny(gray, lower, upper)

# Edge density
edge_density = np.sum(canny_edges > 0) / (H * W)

# ============================================================
# 3. SALIENCY MAP (Attention Prediction)
# ============================================================
# Relates to: G1 (Centroid Wander) - predicting visual attention
saliency_detector = cv2.saliency.StaticSaliencySpectralResidual_create()
success, saliency_map = saliency_detector.computeSaliency(img_bgr)
saliency_map = (saliency_map * 255).astype("uint8")

# Compute saliency-weighted centroid
smap = saliency_map.astype(float) + 1e-8
ys, xs = np.mgrid[0:H, 0:W]
Cx_sal = (xs * smap).sum() / smap.sum()
Cy_sal = (ys * smap).sum() / smap.sum()
Dx_sal = (Cx_sal - W/2) / (W/2)  # normalized displacement

# ============================================================
# 4. DISTANCE TRANSFORM (Void Depth)
# ============================================================
# Relates to: G2 (cut_depth) - measures how deep voids penetrate into mass
# Using inverse Otsu as "voids" (background regions)
void_mask = (otsu_mask == 0).astype(np.uint8)
if np.sum(void_mask) > 0:
    dist_transform = distance_transform_edt(void_mask)
    max_depth = dist_transform.max()
    # Normalize for visualization
    dist_viz = (dist_transform / (max_depth + 1e-6) * 255).astype(np.uint8)
else:
    dist_transform = np.zeros_like(gray)
    max_depth = 0
    dist_viz = dist_transform

# ============================================================
# PLOT 2x2 COMPARISON GRID
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# 1. Otsu Thresholding
axes[0, 0].imshow(otsu_mask, cmap='gray')
axes[0, 0].set_title("1. Otsu Thresholding\n(Figure/Ground Separation)",
                     fontsize=13, weight='bold')
axes[0, 0].axis("off")
axes[0, 0].text(10, 30,
               f"Threshold: {otsu_threshold}\n"
               f"FG ratio: {fg_ratio:.2f}\n"
               f"→ G2 Void Topology",
               color='yellow', fontsize=11, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# 2. Canny Edge Detection
axes[0, 1].imshow(img_rgb)
# Overlay edges in cyan
edge_overlay = np.zeros((*canny_edges.shape, 4))
edge_overlay[..., 0] = 0.0  # R
edge_overlay[..., 1] = 1.0  # G
edge_overlay[..., 2] = 1.0  # B (cyan)
edge_overlay[..., 3] = (canny_edges > 0).astype(float) * 0.8
axes[0, 1].imshow(edge_overlay)
axes[0, 1].set_title("2. Canny Edge Detection\n(Structural Boundaries)",
                     fontsize=13, weight='bold')
axes[0, 1].axis("off")
axes[0, 1].text(10, 30,
               f"Edge density: {edge_density:.4f}\n"
               f"Thresholds: [{lower}, {upper}]\n"
               f"→ G3 Curvature",
               color='white', fontsize=11, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# 3. Saliency Map
axes[1, 0].imshow(img_rgb)
axes[1, 0].imshow(saliency_map, cmap='hot', alpha=0.6)
# Mark saliency centroid
axes[1, 0].scatter([Cx_sal], [Cy_sal], c='cyan', s=200,
                  edgecolor='white', linewidth=2, marker='x')
axes[1, 0].axvline(Cx_sal, color='cyan', linestyle='--', alpha=0.5, linewidth=1.5)
axes[1, 0].set_title("3. Saliency Map\n(Attention Prediction)",
                     fontsize=13, weight='bold')
axes[1, 0].axis("off")
axes[1, 0].text(10, 30,
               f"Centroid Δx: {Dx_sal:.3f}\n"
               f"Position: ({Cx_sal:.0f}, {Cy_sal:.0f})\n"
               f"→ G1 Centroid Wander",
               color='cyan', fontsize=11, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# 4. Distance Transform
axes[1, 1].imshow(dist_viz, cmap='viridis')
axes[1, 1].set_title("4. Distance Transform\n(Void Depth Penetration)",
                     fontsize=13, weight='bold')
axes[1, 1].axis("off")
axes[1, 1].text(10, 30,
               f"Max depth: {max_depth:.1f} px\n"
               f"From void boundaries\n"
               f"→ G2 cut_depth",
               color='yellow', fontsize=11, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.suptitle("Core CV Techniques → VCLI-G Geometric Primitives",
             fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ============================================================
# PRINT INTERPRETATION
# ============================================================
print("\n" + "="*70)
print("HOW THESE CV TECHNIQUES RELATE TO VCLI-G:")
print("="*70)
print()
print("1. OTSU THRESHOLDING → G2 (Void Topology)")
print(f"   • Automatic figure/ground separation at threshold = {otsu_threshold}")
print(f"   • Foreground ratio: {fg_ratio:.2%}")
print("   • VCLI-G uses this to identify voids (breathing zones)")
print("   • Measures: void count, aspect ratio, fragmentation")
print()
print("2. CANNY EDGE DETECTION → G3 (Curvature Torque)")
print(f"   • Clean structural boundaries with edge density = {edge_density:.4f}")
print("   • Adaptive thresholds preserve important edges")
print("   • VCLI-G traces contours from these edges")
print("   • Measures: curvature variance, inflection density")
print()
print("3. SALIENCY MAP → G1 (Centroid Wander)")
print(f"   • Predicts visual attention with centroid Δx = {Dx_sal:.3f}")
print("   • Spectral residual method (frequency domain)")
print("   • VCLI-G uses multi-scale saliency pyramid")
print("   • Measures: path length, curvature across scales")
print()
print("4. DISTANCE TRANSFORM → G2 (cut_depth)")
print(f"   • Deepest void penetration = {max_depth:.1f} pixels")
print("   • Measures how far into voids you can travel from edges")
print("   • VCLI-G uses this to assess void topology complexity")
print("   • Higher depth = more significant negative space")
print()
print("="*70)
print("KEY INSIGHT:")
print("VCLI-G combines these standard CV operations into compositional")
print("primitives that measure spatial intelligence, not just image features.")
print("="*70)


# 5. Four-Panel Gradient / Void / Skeleton Visualization — Documentation

This sequence decomposes the image into the three fundamental gradient-field views from which all kernel metrics (Δx, rᵥ, ρᵣ, μ, xₚ, θ, ds) are derived.  
Each panel exposes a different structural dimension of the same composition.

---

## **1. Original Image (Reference Frame)**  
**What this is:**  
The raw luminance geometry — the unmodified input frame.

**What it shows:**  
- Tonal distribution  
- Subject/void arrangement  
- The baseline against which all gradients are computed

**Kernel connections:**  
- All metrics originate from this frame once converted to grayscale.  
- Provides the coordinate system for Δx, rᵥ, and μ.

---

## **2. Original + Gradient Magnitude (Pressure Field)**  
**What this is:**  
A Sobel-derived pressure field showing where luminance changes most strongly.  
This represents the notebook’s *internal vision* — how the system perceives mass.

**Reveals:**  
- Structural mass  
- Edge pressure  
- Composition torque  
- Hierarchy of visual influence

**Kernel connections:**  
- **Δx** — gradient-weighted centroid  
- **μ** — cohesion via cluster continuity  
- **ρᵣ** — packing density of high-pressure zones  
- **xₚ** — directional imbalance / peripheral pull  
- **θ** — dominant orientation from gradient flow

---

## **3. Original + Void Mask (Breathing Zones)**  
**What this is:**  
Computational segmentation of negative space using adaptive percentile thresholding.

**Reveals:**  
- Breathing zones  
- Low-pressure channels  
- Spatial silence  
- Safe, weak, or empty regions in the composition

**Kernel connections:**  
- **rᵥ** — void ratio directly visualized  
- **μ** — fragmented voids reduce cohesion  
- **xₚ** — void asymmetry shifts perceived pull  
- **ρᵣ** — inverse to void density

---

## **4. Original + Gradient Skeleton (Structural Spine)**  
**What this is:**  
A ridge-trace derived from the Hessian + skeletonization — the topological backbone of the image.

**Reveals:**  
- Structural routing  
- Internal flow  
- Pressure pathways  
- Fracture points  
- How the mass “organizes itself”

**Kernel connections:**  
- **μ** — continuity of skeleton = high cohesion  
- **θ** — torque direction visible in branch orientation  
- **ds** — implied structural thickness from ridge strength + continuity  
- **xₚ** — often aligns with dominant backbone flow

In [ ]:
# ============================================================
# UPLOAD + GENERATE 4-PANEL GRADIENT / VOID / SKELETON VIEWS
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import sobel
from skimage.morphology import skeletonize
from skimage import exposure
from google.colab import files

# ---- Upload image ----
print("Upload an image...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ---- Load + preprocess ----
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not read image: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

# ---- Sobel gradients ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)

# Gradient magnitude (pressure field)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

# ---- Void mask ----
void_thresh = np.percentile(grad_mag, 60)
void_mask = grad_mag < void_thresh

# ---- Skeleton ----
skel = skeletonize(grad_mag_norm > np.percentile(grad_mag_norm, 85))

# ---- PLOT 4-PANEL FIGURE ----
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
ax = axes.ravel()

# 1. Original
ax[0].imshow(img)
ax[0].set_title("Original Image")
ax[0].axis("off")

# 2. Pressure Field
ax[1].imshow(img)
ax[1].imshow(grad_mag_norm, cmap="magma", alpha=0.5)
ax[1].set_title("Original + Gradient Magnitude (Pressure Field)")
ax[1].axis("off")

# 3. Void mask overlay
void_overlay = np.zeros((*void_mask.shape, 4))
void_overlay[..., 2] = 1.0
void_overlay[..., 3] = void_mask.astype(float) * 0.35
ax[2].imshow(img)
ax[2].imshow(void_overlay)
ax[2].set_title("Original + Void Mask (Breathing Zones)")
ax[2].axis("off")

# 4. Skeleton overlay
skel_overlay = np.zeros((*skel.shape, 4))
skel_overlay[..., :3] = 1.0
skel_overlay[..., 3] = skel.astype(float) * 0.9
ax[3].imshow(img)
ax[3].imshow(skel_overlay)
ax[3].set_title("Original + Gradient Skeleton (Structural Spine)")
ax[3].axis("off")

plt.tight_layout()
plt.show()

#6. Full Compositional Overlay

**What This Overlay Is**

This panel renders the entire compositional intelligence of the image onto a single frame. It is the closest visual equivalent to “Sketcher Lens Vision Mode” — the field-level perception of structure, void, and directional pressure used inside VTL.

It shows:
- Δx centroid: Gradient-weighted center of mass. This reveals where the image actually balances — not where it appears centered. (Portraits almost always skew left or right; Δx exposes this instantly.)
- xₚ peripheral-pull vector: A directional arrow showing where the frame “leans.” This reflects the model’s off-center prior — the basin of attraction pulling composition outward.
- Void hulls (blue wash): The breathing zones. This is rᵥ in spatial form, areas of low gradient pressure where nothing structurally competes.
- Gradient islands (cohesion μ evidence): Bright clusters representing where gradients form coherent, repeating structures. High μ = continuous, unified structure. Low μ = fractured or dispersed mass.
- Torque direction / flow orientation: The orientation field showing the image’s directional bias — how structure “turns.”
- Structural backbone (white skeleton): A thin ridge-trace following the strongest gradient attractors. This often maps directly to Sketcher Lens pressure-pathways (A4/A5) and reveals fracture lines.

**What This Overlay Means**
This composite view makes all spatial priors visible at once:

1. Δx: Shows how the image balances mass. If Δx ≠ 0, the composition is structurally asymmetric, even if visually symmetric.
2. xₚ (Peripheral Pull): Shows the direction of the model’s default compositional lean. Often matches MidJourney/Sora/GPT-style left-bias or top-right light bias.
3. rᵥ (Void Ratio): The shape and extent of blue voids give immediate intuition for breathing zones and tension pathways.
4. ρᵣ (Packing Density): Dense gradient islands = compressed structural zones.
5. μ (Cohesion): Whether the structure behaves as one organism or several fragments.
6. θ (Orientation Stability): Global torque direction and the degree to which the composition fights or accepts vertical/horizontal gravity.
7. ds (Structural Thickness): Skeleton thickness approximates the volumetric vs filamentary nature of the mass.

**Why This Overlay Is Valuable**
This overlay is the single unifying visualization for:
- compositional balance
- priors and attractors
- void breathing
- directional pressure
- structural coherence
- pressure fractures
- perceptual torque

It allows you (and reviewers) to see the field the same way the kernel sees it.

In [ ]:
# ==== CELL 1: Upload Image (No Path Needed) ====

!pip install -q scikit-image opencv-python-headless

import cv2, numpy as np, matplotlib.pyplot as plt
from skimage.filters import sobel
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from skimage import exposure

# Upload UI
from google.colab import files

print("Upload an image...")
uploaded = files.upload()

# Automatically store the first uploaded file path
IMAGE_PATH = next(iter(uploaded))
print("Image loaded as:", IMAGE_PATH)

In [ ]:
# ==== CELL 2: Full Compositional Overlay ====

def visualize_full_compositional_overlay(path, figsize=(10, 10)):
    """
    Full compositional overlay on top of the original image:
      - Skeleton of high gradients (structural spine)
      - Void mask (breathing zones)
      - Gradient-mass centroid + Δx
      - Peripheral pull vector xₚ
      - Cohesion μ estimate from gradient islands
    """

    # ---- Load image ----
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise ValueError(f"Could not load image: {path}")

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_f = gray.astype("float32") / 255.0

    H, W = gray.shape

    # ---- Gradient field ----
    gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx**2 + gy**2)
    grad_n = exposure.rescale_intensity(grad, out_range=(0, 1))

    # ---- Skeleton ----
    skel = skeletonize(grad_n > np.percentile(grad_n, 85))

    # ---- Void mask ----
    void_mask = grad < np.percentile(grad, 60)

    # ---- Δx centroid ----
    ys, xs = np.mgrid[0:H, 0:W]
    Gsum = grad.sum() + 1e-9
    Cx = (xs * grad).sum() / Gsum
    Cy = (ys * grad).sum() / Gsum
    Dx = (Cx - W/2) / (W/2)     # normalized displacement

    # ---- xₚ peripheral pull vector ----
    dist_left, dist_right = Cx, W - Cx
    dist_top,  dist_bottom = Cy, H - Cy

    pull_x = dist_left - dist_right
    pull_y = dist_top - dist_bottom

    pull_vec = np.array([pull_x, pull_y], dtype=float)
    if np.linalg.norm(pull_vec) > 1e-6:
        pull_vec /= np.linalg.norm(pull_vec)

    # ---- Cohesion μ ----
    high_mask = grad > np.percentile(grad, 75)
    labeled = label(high_mask)
    props = regionprops(labeled)
    areas = np.array([p.area for p in props]) if len(props) else np.array([1.0])

    p = areas / areas.sum()
    mu = 1 - (-(p * np.log(p + 1e-9)).sum() / np.log(len(p) + 1e-9))

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.set_title("Full Compositional Overlay", fontsize=14)
    ax.axis("off")

    # Skeleton overlay
    skel_overlay = np.zeros((*skel.shape, 4))
    skel_overlay[..., :3] = 1.0
    skel_overlay[..., 3] = skel * 0.8
    ax.imshow(skel_overlay)

    # Void overlay
    void_overlay = np.zeros((*void_mask.shape, 4))
    void_overlay[..., 2] = 1.0
    void_overlay[..., 3] = void_mask * 0.25
    ax.imshow(void_overlay)

    # Centroid
    ax.scatter([Cx], [Cy], c="yellow", s=120, edgecolor="black", linewidth=1.5)
    ax.axvline(Cx, color="yellow", linestyle="--", alpha=0.6)

    # Pull vector arrow
    arrow_len = min(H, W) * 0.15
    ax.arrow(
        Cx, Cy,
        pull_vec[0] * arrow_len,
        pull_vec[1] * arrow_len,
        color="red", width=2, head_width=12,
        alpha=0.9, length_includes_head=True
    )

    # Labels
    ax.text(10, 20, f"Δx = {Dx:.3f}", color="yellow", fontsize=12, weight="bold")
    ax.text(10, 45, f"μ = {mu:.3f}",   color="white",  fontsize=12, weight="bold")
    ax.text(10, 70, f"xₚ → ({pull_vec[0]:.2f}, {pull_vec[1]:.2f})",
            color="red", fontsize=12, weight="bold")

    plt.show()


# ---- Run the overlay on the uploaded image ----
visualize_full_compositional_overlay(IMAGE_PATH)

# 7. Side-by-Side Compositional Comparison  
### Two-Stage Upload • Δx Shift • xₚ Pull • Void Basins • Structural Spine  
Compare two images with full VCLI-G overlays and difference metrics.

---

## 🔍 What this cell does
This module allows you to upload **two separate images** and visualize their full compositional intelligence **side-by-side**.

Typical use cases:
- AI Default vs. Master Artwork  
- Iteration 1 vs. Iteration 5  
- Centered Composition vs. Intentional Off-Center  
- Before vs. After (editing, prompting, or refinement)

---

## 📥 Two-Stage Upload Workflow
### **Image A**
- Usually the *default* or *early iteration*  
- Represents the model’s unintentional spatial prior  
- Serves as the baseline for Δx comparison  

### **Image B**
- Usually the *master work* or *later iteration*  
- Represents the intentional spatial solution  
- Shows how the composition evolves under deliberate pressure  

You upload them **separately** to avoid overwriting errors  
and to make the UI flow intuitive.

---

## 🧩 What each panel displays (per image)

### **1. Yellow Δx Line + Centroid Dot**
- Shows the gradient-weighted mass center  
- Δx displacement is drawn from geometric center → centroid  
- Reveals subtle model bias even when the subject appears centered  

### **2. Red xₚ Vector (Peripheral Pull)**
- Indicates where the composition “leans”  
- Spatial attractor computed from edge-weighted gradient distribution  
- Often exposes hidden directional bias invisible to the naked eye  

### **3. Blue Void Basins (Breathing Zones)**
- Derived from the gradient void mask  
- Highlights negative space that carries structural importance  
- Shows where the composition exerts “release” vs. “pressure”  

### **4. White Structural Skeleton (Backbone)**
- Thinned topology of the gradient field  
- Matches your Sketcher Lens pressure pathways almost 1:1  
- Shows directional flow, fracture lines, and compositional tension  

---

## 📊 Metrics printed for each image  
- **Δx** (centroid displacement)  
- **μ** (cohesion from gradient island stability)  
- **rᵥ** (void ratio)  
- **xₚ vector magnitude + angle**  
- **Structural length measure (skeleton complexity)**  

These values make the two images directly comparable.

---

## 🆚 Difference Analysis (A vs B)
The cell computes:
- Δx shift between images  
- Change in void ratio  
- Cohesion difference (μ₍B₎ − μ₍A₎)  
- Change in skeleton complexity  
- Net directional drift of xₚ  

This gives you a quantitative “where did the composition move?” summary.

---

## 🧠 Why this cell matters  
This is the closest to a **VCLI-G-assisted A/B critique engine**.  
It reveals how two compositions diverge structurally, showing:

- What the model defaults to  
- What your intentional iteration achieves  
- How spatial priors shift across recursion  
- Where pressure increases or dissipates  
- Whether progression is meaningful or cosmetic  

In short:  
**It turns compositional comparison into measurable spatial intelligence.**


In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPOSITIONAL COMPARISON
# Compare two images with full VCLI-G overlays + difference metrics
# Perfect for: AI vs Master, Default vs Intentional, Before vs After
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from google.colab import files

# ============================================================
# UPLOAD TWO IMAGES
# ============================================================
print("="*70)
print("COMPARATIVE ANALYSIS: Upload TWO images to compare")
print("="*70)
print("\n📸 First, upload IMAGE A (e.g., AI default, iteration 1, centered)...")
uploaded_A = files.upload()
IMAGE_A = next(iter(uploaded_A))
print(f"✓ Image A loaded: {IMAGE_A}\n")

print("📸 Now, upload IMAGE B (e.g., master work, iteration 5, intentional)...")
uploaded_B = files.upload()
IMAGE_B = next(iter(uploaded_B))
print(f"✓ Image B loaded: {IMAGE_B}\n")

# ============================================================
# ANALYSIS FUNCTION
# ============================================================
def analyze_composition(path):
    """
    Full compositional analysis returning metrics and visualizations.
    """
    # Load
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise ValueError(f"Could not load: {path}")

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_f = gray.astype("float32") / 255.0

    H, W = gray.shape

    # ---- Gradient field ----
    gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx**2 + gy**2)
    grad_n = exposure.rescale_intensity(grad, out_range=(0, 1))

    # ---- Skeleton ----
    skel = skeletonize(grad_n > np.percentile(grad_n, 85))
    edge_density = skel.sum() / (H * W)

    # ---- Void mask ----
    void_mask = grad < np.percentile(grad, 60)
    void_ratio = void_mask.sum() / (H * W)

    # ---- Δx centroid ----
    ys, xs = np.mgrid[0:H, 0:W]
    Gsum = grad.sum() + 1e-9
    Cx = (xs * grad).sum() / Gsum
    Cy = (ys * grad).sum() / Gsum
    Dx = (Cx - W/2) / (W/2)  # normalized displacement

    # ---- xₚ peripheral pull vector ----
    dist_left, dist_right = Cx, W - Cx
    dist_top, dist_bottom = Cy, H - Cy
    pull_x = dist_left - dist_right
    pull_y = dist_top - dist_bottom
    pull_vec = np.array([pull_x, pull_y], dtype=float)
    if np.linalg.norm(pull_vec) > 1e-6:
        pull_vec /= np.linalg.norm(pull_vec)

    # ---- Cohesion μ ----
    high_mask = grad > np.percentile(grad, 75)
    labeled = label(high_mask)
    props = regionprops(labeled)
    areas = np.array([p.area for p in props]) if len(props) else np.array([1.0])
    p = areas / areas.sum()
    mu = 1 - (-(p * np.log(p + 1e-9)).sum() / np.log(len(p) + 1e-9))

    return {
        'img': img,
        'skel': skel,
        'void_mask': void_mask,
        'Cx': Cx,
        'Cy': Cy,
        'Dx': Dx,
        'pull_vec': pull_vec,
        'mu': mu,
        'void_ratio': void_ratio,
        'edge_density': edge_density,
        'H': H,
        'W': W
    }

# ============================================================
# ANALYZE BOTH IMAGES
# ============================================================
print("Analyzing Image A...")
results_A = analyze_composition(IMAGE_A)
print("Analyzing Image B...")
results_B = analyze_composition(IMAGE_B)

# ============================================================
# COMPUTE DIFFERENCES
# ============================================================
diff_Dx = results_B['Dx'] - results_A['Dx']
diff_void = results_B['void_ratio'] - results_A['void_ratio']
diff_edge = results_B['edge_density'] - results_A['edge_density']
diff_mu = results_B['mu'] - results_A['mu']

# ============================================================
# PLOT SIDE-BY-SIDE COMPARISON
# ============================================================
fig = plt.figure(figsize=(20, 10))

# Create 2 main subplots for the overlays
ax_A = plt.subplot(1, 2, 1)
ax_B = plt.subplot(1, 2, 2)

# ---- IMAGE A OVERLAY ----
ax_A.imshow(results_A['img'])
ax_A.set_title(f"IMAGE A: {IMAGE_A[:30]}", fontsize=14, weight='bold', pad=10)
ax_A.axis("off")

# Skeleton overlay
skel_overlay_A = np.zeros((*results_A['skel'].shape, 4))
skel_overlay_A[..., :3] = 1.0
skel_overlay_A[..., 3] = results_A['skel'] * 0.7
ax_A.imshow(skel_overlay_A)

# Void overlay
void_overlay_A = np.zeros((*results_A['void_mask'].shape, 4))
void_overlay_A[..., 2] = 1.0
void_overlay_A[..., 3] = results_A['void_mask'] * 0.25
ax_A.imshow(void_overlay_A)

# Centroid
ax_A.scatter([results_A['Cx']], [results_A['Cy']],
            c="yellow", s=150, edgecolor="black", linewidth=2, zorder=10)
ax_A.axvline(results_A['Cx'], color="yellow", linestyle="--", alpha=0.6, linewidth=1.5)

# Pull vector
arrow_len_A = min(results_A['H'], results_A['W']) * 0.12
ax_A.arrow(results_A['Cx'], results_A['Cy'],
          results_A['pull_vec'][0] * arrow_len_A,
          results_A['pull_vec'][1] * arrow_len_A,
          color="red", width=2, head_width=10, alpha=0.9,
          length_includes_head=True, zorder=9)

# Labels
ax_A.text(15, 35, f"Δx = {results_A['Dx']:+.3f}",
         color="yellow", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
ax_A.text(15, 70, f"μ = {results_A['mu']:.3f}",
         color="white", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
ax_A.text(15, 105, f"void = {results_A['void_ratio']:.3f}",
         color="cyan", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# ---- IMAGE B OVERLAY ----
ax_B.imshow(results_B['img'])
ax_B.set_title(f"IMAGE B: {IMAGE_B[:30]}", fontsize=14, weight='bold', pad=10)
ax_B.axis("off")

# Skeleton overlay
skel_overlay_B = np.zeros((*results_B['skel'].shape, 4))
skel_overlay_B[..., :3] = 1.0
skel_overlay_B[..., 3] = results_B['skel'] * 0.7
ax_B.imshow(skel_overlay_B)

# Void overlay
void_overlay_B = np.zeros((*results_B['void_mask'].shape, 4))
void_overlay_B[..., 2] = 1.0
void_overlay_B[..., 3] = results_B['void_mask'] * 0.25
ax_B.imshow(void_overlay_B)

# Centroid
ax_B.scatter([results_B['Cx']], [results_B['Cy']],
            c="yellow", s=150, edgecolor="black", linewidth=2, zorder=10)
ax_B.axvline(results_B['Cx'], color="yellow", linestyle="--", alpha=0.6, linewidth=1.5)

# Pull vector
arrow_len_B = min(results_B['H'], results_B['W']) * 0.12
ax_B.arrow(results_B['Cx'], results_B['Cy'],
          results_B['pull_vec'][0] * arrow_len_B,
          results_B['pull_vec'][1] * arrow_len_B,
          color="red", width=2, head_width=10, alpha=0.9,
          length_includes_head=True, zorder=9)

# Labels
ax_B.text(15, 35, f"Δx = {results_B['Dx']:+.3f}",
         color="yellow", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
ax_B.text(15, 70, f"μ = {results_B['mu']:.3f}",
         color="white", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
ax_B.text(15, 105, f"void = {results_B['void_ratio']:.3f}",
         color="cyan", fontsize=13, weight="bold",
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.suptitle("Side-by-Side Compositional Comparison",
            fontsize=18, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ============================================================
# PRINT COMPARATIVE ANALYSIS
# ============================================================
print("\n" + "="*70)
print("COMPARATIVE METRICS: B vs A")
print("="*70)
print()
print(f"{'Metric':<25} {'Image A':<15} {'Image B':<15} {'Δ (B-A)':<15}")
print("-"*70)
print(f"{'Centroid Δx':<25} {results_A['Dx']:+.4f}{'':<10} {results_B['Dx']:+.4f}{'':<10} {diff_Dx:+.4f}")
print(f"{'Void Ratio':<25} {results_A['void_ratio']:.4f}{'':<10} {results_B['void_ratio']:.4f}{'':<10} {diff_void:+.4f}")
print(f"{'Edge Density':<25} {results_A['edge_density']:.4f}{'':<10} {results_B['edge_density']:.4f}{'':<10} {diff_edge:+.4f}")
print(f"{'Cohesion μ':<25} {results_A['mu']:.4f}{'':<10} {results_B['mu']:.4f}{'':<10} {diff_mu:+.4f}")
print()
print("="*70)
print("INTERPRETATION:")
print("="*70)

# Centroid difference
if abs(diff_Dx) < 0.05:
    print("⚖️  Centroid: Both images have similar balance")
elif abs(results_B['Dx']) > abs(results_A['Dx']):
    print(f"↗️  Centroid: Image B is MORE off-center (|Δx| = {abs(results_B['Dx']):.3f})")
    print("   → More intentional compositional bias")
else:
    print(f"↙️  Centroid: Image A is MORE off-center (|Δx| = {abs(results_A['Dx']):.3f})")
    print("   → Image B is more centered/default")

# Void ratio difference
if abs(diff_void) < 0.05:
    print("🔳 Voids: Both images have similar breathing zones")
elif diff_void > 0:
    print(f"🌬️  Voids: Image B has MORE negative space (+{diff_void:.3f})")
    print("   → More breathing room, less compressed")
else:
    print(f"📦 Voids: Image A has MORE negative space ({diff_void:.3f})")
    print("   → Image B is more densely packed")

# Edge density difference
if abs(diff_edge) < 0.001:
    print("📏 Edges: Both images have similar structural density")
elif diff_edge > 0:
    print(f"🕸️  Edges: Image B has MORE skeletal structure (+{diff_edge:.4f})")
    print("   → Richer compositional armature")
else:
    print(f"🕸️  Edges: Image A has MORE skeletal structure ({diff_edge:.4f})")
    print("   → Image B is structurally simpler")

# Cohesion difference
if abs(diff_mu) < 0.05:
    print("🔗 Cohesion: Both images have similar field unity")
elif diff_mu > 0:
    print(f"🎯 Cohesion: Image B is MORE unified (μ +{diff_mu:.3f})")
    print("   → Stronger compositional integration")
else:
    print(f"💥 Cohesion: Image A is MORE unified (μ {diff_mu:.3f})")
    print("   → Image B is more fragmented/emergent")

print()
print("="*70)
print("KEY VISUAL DIFFERENCES:")
print("="*70)
print("Look at the overlays above to see:")
print("  • Yellow centroid + line: Where gradient mass balances")
print("  • Red arrow: Peripheral pull direction (xₚ)")
print("  • Blue wash: Void basins (breathing zones)")
print("  • White tracery: Structural skeleton (compositional spine)")
print()
print("Compositional sophistication shows as:")
print("  ✓ Off-center Δx (intentional bias, not default centering)")
print("  ✓ Rich void topology (breathing zones, not compression)")
print("  ✓ Complex skeleton (structural intelligence, not uniform density)")
print("  ✓ High cohesion μ (unified field, not scattered fragments)")
print("="*70)


# 8. Compositional Fingerprint (Radar Chart)
### The “QR Code” of Composition • G1–G4 Primitives • Automatic Profile Classification

---

## 🔍 What this cell creates
A side-by-side visualization showing:

1. **Original Image**  
2. **Radar/Spider Chart** plotting all four geometric primitives:
   - **G1 – Curvature**
   - **G2 – Void Topology**
   - **G3 – Contour / Occlusion**
   - **G4 – Gradient Islands / Cohesion**

Each axis captures one major dimension of spatial intelligence.  
Together, they form a **compositional fingerprint**.

---

## 🎯 Automatic Profile Classification
The radar chart shape allows immediate interpretation:

| Profile Shape | Meaning | Typical Source |
|---------------|---------|----------------|
| ⚪ **CIRCULAR** | Uniform, low-variance, default structure | *AI defaults / no strategy* |
| ⭐ **SPIKY** | High emphasis on certain primitives, intentional strain | *Masters, intentional compositions* |
| ♦ **MIXED** | Balanced but irregular complexity | *Iterative artwork, partial intention* |

Spiky = human decision-making.  
Circular = machine prior.  
Mixed = transitional intelligence.

---

## 🧠 Why this matters
This fingerprint is effectively a **compressed signature of an image’s compositional intelligence**.

Masters tend to show:
- Strong spikes on G1 (curvature structure)
- Or spikes on G2 (void topology)
- Or spikes on G4 (cohesion / island pressure)

For example:
- **Cézanne** → G1 spike  
- **Rothko** → G2 spike  
- **Dutch portraiture** → G4 spike  

AI defaults cluster as **sad little circles**, indicating uniformity and low intentionality.

This is your quickest way to:
- Compare AI vs human composition  
- Track iteration progress  
- Detect whether structure becomes intentional or collapses into defaults  
- Build datasets of compositional archetypes  

---

## 📦 What’s computed under the hood (G1–G4)
- **G1:** Curvature & contour energy  
- **G2:** Void topology (connected components of negative space)  
- **G3:** Occlusion / edge discontinuity  
- **G4:** Gradient island cohesion (μ analogue)

Each is normalized so the radar chart is comparable across images.

---

## 🖼 Output Overview
You will see:

**Left:** Original image  
**Right:** Radar chart with labeled axes, normalized values, and automatic profile label

You can instantly tell:
- Is this composition intentional?  
- Does it match the master’s structural “voice”?  
- Did the model collapse, stabilize, or drift?

This is the closest you can get to a **single-image compositional diagnosis**.

In [ ]:
# ============================================================
# COMPOSITIONAL FINGERPRINT (Radar Chart)
# Visual signature of geometric primitives G1-G4
# Masters = distinctive spiky profiles, AI defaults = circular blobs
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure
from skimage.filters import sobel
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops, find_contours
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter, distance_transform_edt
from google.colab import files
from math import pi

# ---- Upload image ----
print("Upload an image to generate its compositional fingerprint...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ============================================================
# COMPUTE ALL FOUR GEOMETRIC PRIMITIVES (G1-G4)
# ============================================================

def compute_vclig_primitives(path):
    """
    Compute all four VCLI-G geometric signals.
    Returns normalized values for radar plotting.
    """
    # Load
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise ValueError(f"Could not load: {path}")

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_f = gray.astype("float32") / 255.0

    H, W = gray.shape

    # ============================================================
    # G1: CENTROID WANDER (multi-scale saliency)
    # ============================================================
    sigmas = [1.5, 3, 6, 12, 24]
    centroids = []

    for sigma in sigmas:
        blurred = gaussian_filter(gray_f, sigma=sigma)
        gx = sobel(blurred, axis=1)
        gy = sobel(blurred, axis=0)
        saliency = np.hypot(gx, gy)

        # Centroid
        ys, xs = np.mgrid[0:H, 0:W]
        total = saliency.sum() + 1e-9
        cx = (xs * saliency).sum() / total
        cy = (ys * saliency).sum() / total
        centroids.append([cx, cy])

    centroids = np.array(centroids)

    # Path length (normalized by image diagonal)
    if len(centroids) > 1:
        diffs = np.diff(centroids, axis=0)
        path_length = np.linalg.norm(diffs, axis=1).sum()
        diagonal = np.sqrt(H**2 + W**2)
        G1_path = path_length / diagonal
    else:
        G1_path = 0.0

    # Path curvature (mean turning angle)
    if len(centroids) > 2:
        v = np.diff(centroids, axis=0)
        lens = np.linalg.norm(v, axis=1) + 1e-8
        v = v[lens > 1e-6]
        if len(v) > 1:
            v_norm = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
            dots = np.clip((v_norm[:-1] * v_norm[1:]).sum(axis=1), -1.0, 1.0)
            angles = np.arccos(dots)
            G1_curv = float(np.mean(angles))
        else:
            G1_curv = 0.0
    else:
        G1_curv = 0.0

    # Scale up for visibility in 0-3 range
    G1 = (G1_path * 3.0) + (G1_curv * 2.0)  # Combined signal

    # ============================================================
    # G2: VOID TOPOLOGY
    # ============================================================
    gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx**2 + gy**2)

    # Adaptive threshold for voids
    void_mask = grad < np.percentile(grad, 60)

    # Count voids
    labeled_voids = label(void_mask)
    num_voids = labeled_voids.max()

    # Cut depth (deepest void penetration)
    if num_voids > 0:
        props = regionprops(labeled_voids)
        largest_void = max(props, key=lambda p: p.area)
        void_binary = (labeled_voids == largest_void.label).astype(np.uint8)
        dist_transform = distance_transform_edt(void_binary)
        cut_depth = dist_transform.max()
    else:
        cut_depth = 0.0

    # Normalize with log scaling for interpretability
    G2_voids = np.log1p(num_voids) / 5.0  # Log scale, more conservative
    G2_depth = (cut_depth / min(H, W)) * 1.5  # Relative to image size

    G2 = G2_voids + G2_depth  # Combined signal

    # ============================================================
    # G3: CURVATURE TORQUE
    # ============================================================
    g_smooth = gaussian_filter(gray_f, 1.0)
    level = float(g_smooth.mean())
    contours = find_contours(g_smooth, level=level)

    curvatures = []
    inflections = 0

    for c in contours:
        if len(c) < 9:
            continue
        x = c[:, 1]
        y = c[:, 0]

        k = min(9, len(c) - 1)
        if k % 2 == 0:
            k -= 1
        if k < 5:
            continue

        xs = savgol_filter(x, k, 2, mode='interp')
        ys = savgol_filter(y, k, 2, mode='interp')

        dx = np.gradient(xs)
        dy = np.gradient(ys)
        ddx = np.gradient(dx)
        ddy = np.gradient(dy)

        denom = (dx*dx + dy*dy)**1.5 + 1e-8
        kappa = (dx*ddy - dy*ddx) / denom

        curvatures.append(kappa)
        inflections += int(np.sum(np.sign(kappa[:-1]) != np.sign(kappa[1:])))

    if curvatures:
        curvatures = np.concatenate(curvatures)
        k_var = float(np.var(curvatures))
        infl_density = float(inflections / max(1, len(curvatures)))
    else:
        k_var = 0.0
        infl_density = 0.0

    # Log scale with increased sensitivity
    G3 = (np.log1p(k_var * 100) * 0.4) + (infl_density * 10.0)  # Combined signal

    # ============================================================
    # G4: OCCLUSION ENTROPY (orientation entropy)
    # ============================================================
    angles = np.arctan2(gy, gx)
    mag = np.hypot(gx, gy)

    # Weight by magnitude
    hist, _ = np.histogram(angles[mag > np.percentile(mag, 25)],
                          bins=36, range=(-np.pi, np.pi))

    p = hist / (hist.sum() + 1e-8)
    H_entropy = -np.sum(p * np.log2(p + 1e-12))
    H_max = np.log2(36)

    G4 = (H_entropy / H_max) * 2.5  # Normalized to 0-2.5 range

    # NORMALIZATION STRATEGY (REVISED):
    # Signals rebalanced to distribute across 0-3 range more evenly:
    # - G1: Scaled up 3x (path) + 2x (curv) to make attention instability visible
    # - G2: Log-scaled more conservatively (/5 instead of /4) to prevent dominance
    # - G3: Increased inflection weight (10x) to capture formal tension variations
    # - G4: Moderate scaling (2.5x) to balance with others

    return {
        'img': img,
        'G1': float(G1),
        'G2': float(G2),
        'G3': float(G3),
        'G4': float(G4)
    }

# ============================================================
# ANALYZE IMAGE
# ============================================================
print("Computing geometric primitives...")
results = compute_vclig_primitives(IMAGE_PATH)

G1 = results['G1']
G2 = results['G2']
G3 = results['G3']
G4 = results['G4']

print(f"✓ G1 (Centroid Wander): {G1:.3f}")
print(f"✓ G2 (Void Topology): {G2:.3f}")
print(f"✓ G3 (Curvature Torque): {G3:.3f}")
print(f"✓ G4 (Occlusion Entropy): {G4:.3f}")

# ============================================================
# PLOT COMPOSITIONAL FINGERPRINT (Radar Chart)
# ============================================================
fig = plt.figure(figsize=(14, 7))

# Left: Original image
ax_img = plt.subplot(1, 2, 1)
ax_img.imshow(results['img'])
ax_img.set_title(f"Original: {IMAGE_PATH[:40]}", fontsize=12, weight='bold')
ax_img.axis('off')

# Right: Radar chart
ax_radar = plt.subplot(1, 2, 2, projection='polar')

# Categories
categories = ['G1\nCentroid\nWander', 'G2\nVoid\nTopology',
              'G3\nCurvature\nTorque', 'G4\nOcclusion\nEntropy']
N = len(categories)

# Values
values = [G1, G2, G3, G4]
values += values[:1]  # Close the polygon

# Angles
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Calculate dynamic y-axis range first
max_val = max(values[:-1])  # Exclude duplicate closing value
y_max = max(3.5, max_val * 1.3)  # At least 3.5, or 30% above max value

# Reference circles for scale (draw first so they're behind the data)
circle_step = y_max / 6
for i in range(1, 7):
    r = circle_step * i
    ax_radar.plot(angles, [r]*len(angles), '--', color='gray',
                 alpha=0.3, linewidth=0.5)

# Plot main data on top
ax_radar.plot(angles, values, 'o-', linewidth=3, color='#FF6B6B',
             markersize=10, label='Measured')
ax_radar.fill(angles, values, alpha=0.25, color='#FF6B6B')

# Styling
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, size=11, weight='bold')

# Set y-axis limits (already calculated above)
ax_radar.set_ylim(0, y_max)

# Generate dynamic tick marks
if y_max <= 4.0:
    ticks = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
    tick_labels = ['0.5', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5']
else:
    # For higher values, space ticks more broadly
    n_ticks = 6
    tick_step = y_max / n_ticks
    ticks = [tick_step * i for i in range(1, n_ticks + 1)]
    tick_labels = [f'{t:.1f}' for t in ticks]

ax_radar.set_yticks(ticks)
ax_radar.set_yticklabels(tick_labels, size=9, color='gray')
ax_radar.grid(True, alpha=0.3)
ax_radar.set_title('Compositional Fingerprint\n(Log-normalized, dynamic scale)',
                  size=13, weight='bold', pad=20)

plt.suptitle('VCLI-G Geometric Signature', fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ============================================================
# INTERPRETATION
# ============================================================
print("\n" + "="*70)
print("COMPOSITIONAL FINGERPRINT INTERPRETATION:")
print("="*70)
print()
print("GEOMETRIC SIGNALS (normalized 0-3 range):")
print(f"  G1 (Centroid Wander):  {G1:.3f}")
print(f"      Path length + trajectory curvature across scales")
print(f"  G2 (Void Topology):    {G2:.3f}")
print(f"      Log(void count) + void depth (breathing zones)")
print(f"  G3 (Curvature Torque): {G3:.3f}")
print(f"      Log(curvature variance) + inflection density")
print(f"  G4 (Occlusion Entropy):{G4:.3f}")
print(f"      Orientation entropy (depth ambiguity)")
print()
print("PROFILE TYPE:")

# Determine profile type
max_signal = max(G1, G2, G3, G4)
min_signal = min(G1, G2, G3, G4)
variance = np.var([G1, G2, G3, G4])
signal_range = max_signal - min_signal

if variance < 0.2 or signal_range < 0.5:
    profile_type = "⚪ CIRCULAR (Default/Uniform)"
    interpretation = "All signals similar - may indicate default composition or intentional simplicity"
elif signal_range > 1.5 and variance > 0.5:
    profile_type = "⭐ SPIKY (Distinctive Strategy)"
    interpretation = "Strong variation - indicates intentional compositional approach"
    dominant = ['G1 (Centroid Wander)', 'G2 (Void Topology)',
                'G3 (Curvature Torque)', 'G4 (Occlusion Entropy)']
    max_idx = [G1, G2, G3, G4].index(max_signal)
    interpretation += f"\n  → Dominant: {dominant[max_idx]}"

    # Describe what high values mean for each primitive
    if max_idx == 0:
        interpretation += " (dynamic visual attention)"
    elif max_idx == 1:
        interpretation += " (rich breathing zones)"
    elif max_idx == 2:
        interpretation += " (high formal tension)"
    else:
        interpretation += " (complex spatial depth)"
else:
    profile_type = "◆ MIXED (Balanced Complexity)"
    interpretation = "Moderate variation - some sophistication across primitives"

print(f"  {profile_type}")
print(f"  {interpretation}")
print()
print("="*70)
print("WHAT THIS MEANS:")
print("="*70)
print(f"• Chart scale is dynamic (0 to {y_max:.1f}) to fit your data")
print("• All signals rebalanced to distribute across range more evenly")
print("• G2 and G3 use log scaling (exponential variance)")
print("• G1 and G3 scaled up to increase sensitivity to variations")
print("• Masters often show SPIKY profiles - intentional emphasis on specific primitives")
print("• AI defaults tend toward CIRCULAR profiles - uniform mediocrity across all signals")
print("• Compare multiple images to see strategic differences")
print()
print("NORMALIZATION DETAILS (REVISED):")
print("  G1: Path×3 + Curvature×2 - captures attention instability")
print("  G2: Log(voids)/5 + depth×1.5 - breathing zones without dominance")
print("  G3: Log(var)×0.4 + inflections×10 - emphasizes formal tension")
print("  G4: Entropy×2.5 - moderate depth ambiguity")
print()
print("This radar chart is a 'QR code' for compositional intelligence -")
print("distinctive shapes reveal distinctive spatial thinking.")
print("="*70)


# 9. Weighted Attention Heatmap (2×2 Diagnostic Grid)
### Composite “compositional power” map combining G1–G4 primitives

---

## 💡 What this cell generates
A four-panel diagnostic grid:

1. **Original Image**
2. **Individual G-signals** shown as small multiples:
   - **G1 – Curvature field**
   - **G2 – Void topology proximity**
   - **G3 – Occlusion / contour breaks**
   - **G4 – Gradient cohesion**
3. **Composite Heatmap**
   A weighted fusion of all four signals:
   - 35% gradient magnitude  
   - 25% void proximity  
   - 20% curvature  
   - 20% occlusion  
4. **Overlay on the Original**
   Highlights where all signals *agree* — where true compositional intelligence concentrates.

Hot zones (top 10% weighted intensity) are shown in **yellow**.

---

## 🎯 Why this is powerful
This map reveals **where the four geometric primitives align**.  
That alignment is the signature of intentional composition.

- In a **Rembrandt**, hot zones cluster on faces, hands, and major diagonals (deliberate emphasis).  
- In **AI defaults**, hot zones collapse into vague center-mass blobs (accidental symmetry).  
- In iterative work, you can track whether the model begins forming intentional clusters or stays in uniform noise.

This is your most direct measurement of **spatial intention**, because each primitive contributes something different:
- **Gradient** → structural pressure  
- **Void** → breathing zones & negative space logic  
- **Curvature** → flow & directionality  
- **Occlusion** → figure/ground decisions  

Where these overlap, composition becomes *authored*.

---

## ⚙️ Customization
You can tune weights to emphasize different signals depending on:
- Portrait vs landscape  
- Highly gestural vs highly geometric subject matter  
- Diagnostic vs artistic exploration  

Simply adjust the weights vector in the cell below.

---

In [ ]:
# ============================================================
# WEIGHTED ATTENTION HEATMAP
# Composite "compositional power" map combining all G-signals
# Shows where gradient, void, curvature, and occlusion align
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure
from skimage.filters import sobel
from skimage.measure import find_contours
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter, distance_transform_edt
from google.colab import files

# ---- Upload image ----
print("Upload an image to generate weighted attention heatmap...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ============================================================
# LOAD AND PREPROCESS
# ============================================================
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not load: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

print(f"Image size: {W}×{H}")
print("Computing attention fields...")

# ============================================================
# COMPUTE INDIVIDUAL ATTENTION FIELDS
# ============================================================

# ---- G1: GRADIENT MAGNITUDE (baseline attention) ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

field_G1 = grad_mag_norm
print("✓ G1: Gradient magnitude field")

# ---- G2: VOID PROXIMITY (distance from negative space) ----
void_mask = grad_mag < np.percentile(grad_mag, 60)

# Distance from void boundaries (inverted - closer to voids = higher attention)
if void_mask.sum() > 0:
    # Distance from edges of voids
    void_edges = void_mask.astype(np.uint8)
    void_edges = cv2.morphologyEx(void_edges, cv2.MORPH_GRADIENT,
                                   np.ones((3,3), np.uint8))

    # Distance transform from void edges
    dist_from_voids = distance_transform_edt(1 - void_edges)

    # Invert and normalize (closer to voids = higher value)
    max_dist = dist_from_voids.max()
    if max_dist > 0:
        field_G2 = 1.0 - (dist_from_voids / max_dist)
    else:
        field_G2 = np.ones_like(gray_f) * 0.5
else:
    field_G2 = np.ones_like(gray_f) * 0.5

print("✓ G2: Void proximity field")

# ---- G3: CURVATURE DENSITY (local structural complexity) ----
# Use Laplacian as proxy for curvature/corner density
laplacian = cv2.Laplacian(gray_f, cv2.CV_32F, ksize=3)
laplacian_abs = np.abs(laplacian)

# Smooth and normalize
field_G3_raw = gaussian_filter(laplacian_abs, sigma=2.0)
field_G3 = exposure.rescale_intensity(field_G3_raw, out_range=(0, 1))

print("✓ G3: Curvature density field")

# ---- G4: OCCLUSION ENTROPY (local orientation complexity) ----
# Compute local orientation entropy using sliding window
def local_orientation_entropy(gx, gy, window_size=15):
    """
    Compute local orientation entropy at each pixel.
    High entropy = many edge directions = potential occlusion.
    """
    H, W = gx.shape
    entropy_map = np.zeros((H, W), dtype=np.float32)

    half_w = window_size // 2
    mag = np.hypot(gx, gy)
    angles = np.arctan2(gy, gx)

    # Quantize angles into 8 bins
    angle_bins = ((angles + np.pi) / (2 * np.pi) * 8).astype(int) % 8

    for i in range(half_w, H - half_w, 4):  # Stride for speed
        for j in range(half_w, W - half_w, 4):
            # Extract local window
            window_mag = mag[i-half_w:i+half_w+1, j-half_w:j+half_w+1]
            window_bins = angle_bins[i-half_w:i+half_w+1, j-half_w:j+half_w+1]

            # Weight by magnitude
            weights = window_mag.flatten()
            bins = window_bins.flatten()

            # Histogram
            hist = np.zeros(8)
            for b, w in zip(bins, weights):
                hist[b] += w

            # Entropy
            p = hist / (hist.sum() + 1e-8)
            H_local = -np.sum(p * np.log2(p + 1e-12))

            # Fill neighborhood (since we're striding)
            entropy_map[i-2:i+3, j-2:j+3] = H_local

    return entropy_map

field_G4_raw = local_orientation_entropy(gx, gy, window_size=15)
field_G4 = exposure.rescale_intensity(field_G4_raw, out_range=(0, 1))

print("✓ G4: Occlusion entropy field")

# ============================================================
# COMBINE INTO WEIGHTED ATTENTION HEATMAP
# ============================================================
print("\nCombining attention fields...")

# Weighted combination (adjustable weights)
w1 = 0.35  # Gradient magnitude (baseline)
w2 = 0.25  # Void proximity
w3 = 0.20  # Curvature density
w4 = 0.20  # Occlusion entropy

attention_heatmap = (w1 * field_G1 +
                     w2 * field_G2 +
                     w3 * field_G3 +
                     w4 * field_G4)

# Normalize to 0-1
attention_heatmap = (attention_heatmap - attention_heatmap.min()) / \
                    (attention_heatmap.max() - attention_heatmap.min() + 1e-8)

print("✓ Composite attention heatmap generated")

# ============================================================
# IDENTIFY HOT ZONES (top 10% attention)
# ============================================================
threshold = np.percentile(attention_heatmap, 90)
hot_zones = attention_heatmap > threshold
hot_zone_pixels = hot_zones.sum()
hot_zone_ratio = hot_zone_pixels / (H * W)

print(f"\n Hot zones (>90th percentile): {hot_zone_ratio:.2%} of image")

# ============================================================
# PLOT 2×2 GRID: ORIGINAL + FIELDS + HEATMAP + OVERLAY
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# 1. Original Image
axes[0, 0].imshow(img)
axes[0, 0].set_title("Original Image", fontsize=13, weight='bold')
axes[0, 0].axis('off')

# 2. Individual Fields (small multiples)
ax_fields = axes[0, 1]
ax_fields.axis('off')
ax_fields.set_title("Individual Attention Fields", fontsize=13, weight='bold')

# Create 2x2 grid of fields within this subplot
field_h = H // 3
field_w = W // 3

combined_fields = np.zeros((H*2, W*2), dtype=np.float32)
combined_fields[0:H, 0:W] = field_G1
combined_fields[0:H, W:W*2] = field_G2
combined_fields[H:H*2, 0:W] = field_G3
combined_fields[H:H*2, W:W*2] = field_G4

# Resize for display
combined_display = cv2.resize(combined_fields, (W, H))
ax_fields.imshow(combined_display, cmap='hot')

# Add labels
label_size = 10
axes[0, 1].text(W*0.25, 30, "G1: Gradient", ha='center',
               color='white', fontsize=label_size, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
axes[0, 1].text(W*0.75, 30, "G2: Void Prox", ha='center',
               color='white', fontsize=label_size, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
axes[0, 1].text(W*0.25, H-20, "G3: Curvature", ha='center',
               color='white', fontsize=label_size, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
axes[0, 1].text(W*0.75, H-20, "G4: Occlusion", ha='center',
               color='white', fontsize=label_size, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# 3. Composite Heatmap Alone
axes[1, 0].imshow(attention_heatmap, cmap='inferno')
axes[1, 0].set_title("Weighted Attention Heatmap\n(Combined G1-G4)",
                     fontsize=13, weight='bold')
axes[1, 0].axis('off')

# Add colorbar
from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(axes[1, 0])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(axes[1, 0].images[0], cax=cax)
cbar.set_label('Compositional Power', rotation=270, labelpad=20, fontsize=11)

# 4. Original + Heatmap Overlay
axes[1, 1].imshow(img)
axes[1, 1].imshow(attention_heatmap, cmap='hot', alpha=0.6)

# Mark hot zones
hot_zone_contours = np.zeros((*hot_zones.shape, 4))
hot_zone_contours[..., 0] = 1.0  # Red
hot_zone_contours[..., 1] = 1.0  # + Green = Yellow
hot_zone_contours[..., 3] = hot_zones.astype(float) * 0.4

axes[1, 1].imshow(hot_zone_contours)
axes[1, 1].set_title("Overlay: Hot Zones (Top 10%)",
                     fontsize=13, weight='bold')
axes[1, 1].axis('off')

axes[1, 1].text(10, 30,
               f"Hot zones: {hot_zone_ratio:.1%}\n"
               f"Weights: G1={w1}, G2={w2}, G3={w3}, G4={w4}",
               color='yellow', fontsize=10, weight='bold',
               bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.suptitle("Weighted Attention Heatmap: Where Compositional Power Concentrates",
             fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ============================================================
# INTERPRETATION
# ============================================================
print("\n" + "="*70)
print("WEIGHTED ATTENTION HEATMAP INTERPRETATION:")
print("="*70)
print()
print("WHAT THIS SHOWS:")
print("  • HOT (red/yellow) = All four primitives align here")
print("  • COOL (blue/black) = Low compositional activity")
print()
print("FIELD CONTRIBUTIONS:")
print(f"  • G1 (Gradient Magnitude):     {w1*100:.0f}% weight")
print(f"  • G2 (Void Proximity):         {w2*100:.0f}% weight")
print(f"  • G3 (Curvature Density):      {w3*100:.0f}% weight")
print(f"  • G4 (Occlusion Entropy):      {w4*100:.0f}% weight")
print()
print(f"HOT ZONE COVERAGE: {hot_zone_ratio:.1%} of image")

if hot_zone_ratio < 0.05:
    print("  → Very sparse - compositional power highly concentrated")
elif hot_zone_ratio < 0.15:
    print("  → Focused - clear areas of compositional emphasis")
elif hot_zone_ratio < 0.30:
    print("  → Distributed - multiple centers of compositional power")
else:
    print("  → Diffuse - compositional activity spread throughout")

print()
print("="*70)
print("KEY INSIGHT:")
print("="*70)
print("Where all four geometric primitives align = where compositional")
print("intelligence is concentrated. In master works, hot zones often")
print("correspond to:")
print("  • Key visual anchors (faces, horizon lines)")
print("  • Compositional pivots (where mass and void meet)")
print("  • Areas of maximum perceptual tension")
print()
print("In AI defaults, hot zones tend to be centered and uniform,")
print("lacking strategic placement.")
print("="*70)


# 10. Quadrant Weight Distribution (3×3 Grid Analysis)
### 3×3 compositional census — gradient mass, void ratio, edge density per zone

This cell slices the image into a **3×3 grid** and measures how much structural “stuff” lives in each zone.  
It reveals whether an image is **center-dumped** (AI default) or **strategically biased** (intentional composition).

---

## 🔍 What the 2×3 grid shows

### 🔼 Top row (3 panels)
1. **Original with 3×3 Grid Overlay**  
   - Visual reference with labeled zones (Top-Left, Center, Bottom-Right, etc.)
   - Helps you see where subjects sit relative to the grid.

2. **Gradient Mass Heatmap**  
   - Each cell shows **% of total gradient magnitude** in that zone.  
   - Red = high structural load, blue = low.  
   - This is where **Δx, ρᵣ, μ, xₚ** all concentrate.

3. **Void Ratio Heatmap**  
   - Each cell shows **void percentage** (low gradient, breathing space).  
   - Blue = high void (more breathing), red = low void (more clutter).  
   - Direct readout of **rᵥ** per zone.

---

### 🔽 Bottom row (3 panels)
4. **Edge Density Heatmap**  
   - Edge count / gradient density per zone.  
   - Green = structurally dense, darker = sparse.  
   - Highlights where the image is “busy” vs calm.

5. **Horizontal Bias Bars (Left–Center–Right)**  
   - Bar chart of gradient mass across columns.  
   - Shows whether weight clusters left, right, or in the center.

6. **Vertical Bias Bars (Top–Center–Bottom)**  
   - Bar chart of gradient mass across rows.  
   - Reveals whether the image sinks, floats, or balances vertically.

---

## 📊 Key Metrics

### **Center Dominance** — % of gradient mass in the center 3×3 cell

Suggested interpretation bands:

- **≥ 20%** → **AI default / compressed**  
  Center doing too much work; likely “safe” model prior.
- **12–20%** → **Moderate / balanced**  
  Some center emphasis, but mass is reasonably distributed.
- **< 12%** → **Distributed / sophisticated**  
  Composition uses off-center zones structurally and intentionally.

---

### **Asymmetry Score** — combined horizontal + vertical imbalance

- **0.0 = perfectly symmetric**  
- **1.0 = extreme asymmetry**

Interpretation bands:

- **< 0.15** → **Low (safe/default)**  
  Symmetric, stable, likely model prior rather than a compositional decision.
- **0.15–0.35** → **Moderate (some intent)**  
  Noticeable bias, often where human editing has begun.
- **> 0.35** → **High (strategic bias)**  
  Strong off-center strategy; common in master works and intentional framing.

---

## 🧠 Why this is powerful

This analysis reveals **spatial strategy immediately**:

- AI defaults tend to:
  - Cluster **20–25%+** of gradient mass in the center cell  
  - Show low asymmetry  
  - Keep left/right and top/bottom fairly uniform

- Intentional compositions (e.g., strong painting, photography, film stills) often:
  - Reduce center dominance  
  - Bias weight into specific zones (e.g., lower-left for stability, upper-right for tension)  
  - Show clear horizontal or vertical asymmetry that matches narrative or visual intent

The heatmaps make the pattern *visible*.  
The bars and metrics make it *measurable*.  
Together they tell you whether compositional mass is **strategically distributed** or just **dumped into the middle**.

Run this on any image and you’ll know, in one glance, whether the structure is **prior-driven** or **authored**.


In [ ]:
# ============================================================
# QUADRANT WEIGHT DISTRIBUTION (3x3 Grid Analysis)
# Shows gradient mass, void ratio, edge density per zone
# Reveals compositional bias - masters create asymmetry, AI clusters center
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from skimage import exposure
from google.colab import files

# ---- Upload image ----
print("Upload an image to analyze quadrant weight distribution...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ============================================================
# LOAD AND PREPROCESS
# ============================================================
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not load: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

print(f"Image size: {W}×{H}")

# ---- Compute gradient field ----
gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

# ============================================================
# DIVIDE INTO 3x3 GRID AND COMPUTE METRICS
# ============================================================
# Zone labels (reading order)
zone_labels = [
    ['Top-Left', 'Top-Center', 'Top-Right'],
    ['Mid-Left', 'Center', 'Mid-Right'],
    ['Bottom-Left', 'Bottom-Center', 'Bottom-Right']
]

# Grid boundaries
row_boundaries = [0, H//3, 2*H//3, H]
col_boundaries = [0, W//3, 2*W//3, W]

# Storage for metrics
metrics = {
    'gradient_mass': np.zeros((3, 3)),
    'void_ratio': np.zeros((3, 3)),
    'edge_density': np.zeros((3, 3))
}

print("\nAnalyzing 3×3 grid zones...")

for i in range(3):
    for j in range(3):
        # Extract zone
        r_start, r_end = row_boundaries[i], row_boundaries[i+1]
        c_start, c_end = col_boundaries[j], col_boundaries[j+1]

        zone = grad_mag_norm[r_start:r_end, c_start:c_end]

        # Metric 1: Gradient Mass (sum of gradient magnitude)
        gradient_mass = zone.sum()

        # Metric 2: Void Ratio (percentage of low-gradient pixels)
        void_threshold = np.percentile(grad_mag_norm, 40)
        voids = (zone < void_threshold).sum()
        void_ratio = voids / zone.size

        # Metric 3: Edge Density (percentage of high-gradient pixels)
        edge_threshold = np.percentile(grad_mag_norm, 75)
        edges = (zone > edge_threshold).sum()
        edge_density = edges / zone.size

        # Store
        metrics['gradient_mass'][i, j] = gradient_mass
        metrics['void_ratio'][i, j] = void_ratio
        metrics['edge_density'][i, j] = edge_density

        print(f"  {zone_labels[i][j]:15} | Mass: {gradient_mass:6.1f} | "
              f"Void: {void_ratio:.2f} | Edge: {edge_density:.2f}")

# ============================================================
# NORMALIZE FOR VISUALIZATION (percentage of total)
# ============================================================
gradient_mass_pct = (metrics['gradient_mass'] / metrics['gradient_mass'].sum()) * 100

# ============================================================
# COMPUTE ASYMMETRY METRICS
# ============================================================
# Center dominance
center_mass = gradient_mass_pct[1, 1]

# Horizontal asymmetry (left vs right)
left_mass = gradient_mass_pct[:, 0].sum()
right_mass = gradient_mass_pct[:, 2].sum()
h_asymmetry = abs(left_mass - right_mass) / (left_mass + right_mass)

# Vertical asymmetry (top vs bottom)
top_mass = gradient_mass_pct[0, :].sum()
bottom_mass = gradient_mass_pct[2, :].sum()
v_asymmetry = abs(top_mass - bottom_mass) / (top_mass + bottom_mass)

# Overall asymmetry score
asymmetry_score = (h_asymmetry + v_asymmetry) / 2

print(f"\n{'='*70}")
print(f"COMPOSITIONAL BIAS METRICS:")
print(f"{'='*70}")
print(f"Center dominance:      {center_mass:.1f}% of total gradient mass")
print(f"Horizontal asymmetry:  {h_asymmetry:.3f} (0=balanced, 1=extreme)")
print(f"Vertical asymmetry:    {v_asymmetry:.3f} (0=balanced, 1=extreme)")
print(f"Overall asymmetry:     {asymmetry_score:.3f}")

# ============================================================
# PLOT 2x3 GRID OF HEATMAPS
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ---- ROW 1: ORIGINAL + GRADIENT MASS + VOID RATIO ----

# 1. Original with 3x3 grid overlay
axes[0, 0].imshow(img)
axes[0, 0].set_title("Original Image\n(3×3 Grid Overlay)", fontsize=13, weight='bold')
axes[0, 0].axis('off')

# Draw grid lines
for r in row_boundaries[1:-1]:
    axes[0, 0].axhline(r, color='yellow', linewidth=2, alpha=0.7)
for c in col_boundaries[1:-1]:
    axes[0, 0].axvline(c, color='yellow', linewidth=2, alpha=0.7)

# Label zones
for i in range(3):
    for j in range(3):
        r_center = (row_boundaries[i] + row_boundaries[i+1]) / 2
        c_center = (col_boundaries[j] + col_boundaries[j+1]) / 2
        axes[0, 0].text(c_center, r_center, zone_labels[i][j],
                       ha='center', va='center', color='yellow',
                       fontsize=10, weight='bold',
                       bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# 2. Gradient Mass Heatmap (% of total)
im1 = axes[0, 1].imshow(gradient_mass_pct, cmap='Reds', vmin=0, vmax=25)
axes[0, 1].set_title("Gradient Mass Distribution\n(% of Total)", fontsize=13, weight='bold')
axes[0, 1].set_xticks([0, 1, 2])
axes[0, 1].set_xticklabels(['Left', 'Center', 'Right'])
axes[0, 1].set_yticks([0, 1, 2])
axes[0, 1].set_yticklabels(['Top', 'Middle', 'Bottom'])

# Annotate with values
for i in range(3):
    for j in range(3):
        val = gradient_mass_pct[i, j]
        color = 'white' if val > 15 else 'black'
        axes[0, 1].text(j, i, f'{val:.1f}%',
                       ha='center', va='center',
                       color=color, fontsize=12, weight='bold')

plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

# 3. Void Ratio Heatmap
im2 = axes[0, 2].imshow(metrics['void_ratio'], cmap='Blues', vmin=0, vmax=1)
axes[0, 2].set_title("Void Ratio\n(Breathing Zones)", fontsize=13, weight='bold')
axes[0, 2].set_xticks([0, 1, 2])
axes[0, 2].set_xticklabels(['Left', 'Center', 'Right'])
axes[0, 2].set_yticks([0, 1, 2])
axes[0, 2].set_yticklabels(['Top', 'Middle', 'Bottom'])

# Annotate with values
for i in range(3):
    for j in range(3):
        val = metrics['void_ratio'][i, j]
        color = 'white' if val > 0.5 else 'black'
        axes[0, 2].text(j, i, f'{val:.2f}',
                       ha='center', va='center',
                       color=color, fontsize=12, weight='bold')

plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

# ---- ROW 2: EDGE DENSITY + BIAS VISUALIZATIONS ----

# 4. Edge Density Heatmap
im3 = axes[1, 0].imshow(metrics['edge_density'], cmap='Greens', vmin=0, vmax=0.5)
axes[1, 0].set_title("Edge Density\n(Structural Concentration)", fontsize=13, weight='bold')
axes[1, 0].set_xticks([0, 1, 2])
axes[1, 0].set_xticklabels(['Left', 'Center', 'Right'])
axes[1, 0].set_yticks([0, 1, 2])
axes[1, 0].set_yticklabels(['Top', 'Middle', 'Bottom'])

# Annotate with values
for i in range(3):
    for j in range(3):
        val = metrics['edge_density'][i, j]
        color = 'white' if val > 0.25 else 'black'
        axes[1, 0].text(j, i, f'{val:.2f}',
                       ha='center', va='center',
                       color=color, fontsize=12, weight='bold')

plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

# 5. Horizontal Bias (Left vs Right)
h_bars = [left_mass, gradient_mass_pct[1, 1], right_mass]
h_labels = ['Left\n3 zones', 'Center\nzone', 'Right\n3 zones']
h_colors = ['#FF6B6B', '#FFD93D', '#6BCF7F']

axes[1, 1].barh(h_labels, h_bars, color=h_colors, edgecolor='black', linewidth=2)
axes[1, 1].set_xlabel('% of Total Gradient Mass', fontsize=11, weight='bold')
axes[1, 1].set_title(f"Horizontal Distribution\n(Asymmetry: {h_asymmetry:.3f})",
                     fontsize=13, weight='bold')
axes[1, 1].set_xlim(0, max(50, max(h_bars) * 1.1))

# Annotate bars
for i, val in enumerate(h_bars):
    axes[1, 1].text(val + 1, i, f'{val:.1f}%',
                   va='center', fontsize=11, weight='bold')

# 6. Vertical Bias (Top vs Bottom)
v_bars = [top_mass, gradient_mass_pct[1, 1], bottom_mass]
v_labels = ['Top\n3 zones', 'Center\nzone', 'Bottom\n3 zones']
v_colors = ['#FF6B6B', '#FFD93D', '#6BCF7F']

axes[1, 2].barh(v_labels, v_bars, color=v_colors, edgecolor='black', linewidth=2)
axes[1, 2].set_xlabel('% of Total Gradient Mass', fontsize=11, weight='bold')
axes[1, 2].set_title(f"Vertical Distribution\n(Asymmetry: {v_asymmetry:.3f})",
                     fontsize=13, weight='bold')
axes[1, 2].set_xlim(0, max(50, max(v_bars) * 1.1))

# Annotate bars
for i, val in enumerate(v_bars):
    axes[1, 2].text(val + 1, i, f'{val:.1f}%',
                   va='center', fontsize=11, weight='bold')

plt.suptitle("Quadrant Weight Distribution: Compositional Bias Analysis",
             fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

# ============================================================
# INTERPRETATION
# ============================================================
print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)

# Center dominance assessment
if center_mass > 20:
    print(f"⚠️  HIGH CENTER DOMINANCE ({center_mass:.1f}%)")
    print("   → Compositional mass clustered in center zone")
    print("   → Typical of AI defaults or amateur composition")
elif center_mass > 12:
    print(f"◆  MODERATE CENTER WEIGHT ({center_mass:.1f}%)")
    print("   → Balanced center with peripheral activity")
    print("   → May indicate intentional centering or moderate skill")
else:
    print(f"✓  DISTRIBUTED COMPOSITION ({center_mass:.1f}%)")
    print("   → Center is not dominant - mass spread across zones")
    print("   → Typical of intentional compositional strategy")

print()

# Asymmetry assessment
if asymmetry_score < 0.15:
    print(f"⚪ LOW ASYMMETRY ({asymmetry_score:.3f})")
    print("   → Nearly symmetric distribution")
    print("   → May indicate default/safe composition or intentional balance")
elif asymmetry_score < 0.35:
    print(f"◆  MODERATE ASYMMETRY ({asymmetry_score:.3f})")
    print("   → Some directional bias present")
    print("   → Suggests compositional intent")
else:
    print(f"⭐ HIGH ASYMMETRY ({asymmetry_score:.3f})")
    print("   → Strong directional bias")
    print("   → Indicates deliberate compositional strategy")

print()
print("="*70)
print("KEY PATTERNS:")
print("="*70)
print("MASTERS typically show:")
print("  • Center dominance < 15% (distributed mass)")
print("  • Asymmetry > 0.25 (intentional directional bias)")
print("  • Uneven zone weights (strategic placement)")
print()
print("AI DEFAULTS typically show:")
print("  • Center dominance > 20% (compressed composition)")
print("  • Asymmetry < 0.15 (safe/symmetric)")
print("  • Uniform peripheral zones (no strategic variation)")
print()
print("This analysis reveals WHERE compositional intelligence is placed,")
print("not just IF it exists. Strategic asymmetry = intentional thinking.")
print("="*70)

# 11. Cohesion Visual Companion (μ)  
### Gradient Field → Topology → Entropy → Metric  
A complete pipeline showing how μ (cohesion) is actually computed — from pixels to perceptual measure.

This 3×2 grid walks through the entire topological process that converts raw gradients into an interpretable structural metric.  
No heuristics. No subjectivity. Just gradient behavior.

---

## 🔍 What the 3×2 Grid Shows

### **Row 1 — From Image to Forces**
1. **Original Image (Reference)**  
   The raw frame, untouched. Baseline for visual comparison.

2. **Gradient Magnitude Field \|∇I\| (Pressure Map)**  
   Hot colormap showing where edges, transitions, and structural energy concentrate.  
   This is the “force surface” from which cohesion emerges.

---

### **Row 2 — Extracting Structural Units**
3. **Thresholded High-Gradient Mask (top 25%)**  
   Removes noise; keeps only meaningful structural signal.  
   This is the same philosophy underlying VCLI-G: prioritize strong gradients.

4. **Connected Components → “Gradient Islands”**  
   Each island receives a unique color.  
   These are not semantic regions — they are **topological clusters of structural energy**.  
   Example fields:  
   - Number of islands (K): 47  
   - Island sizes: min=8 px, max=12,847 px, mean=324 px

---

### **Row 3 — From Distribution to Measurement**
5. **Top 5 Largest Islands Highlighted**  
   Largest = red, next = orange, etc.  
   Makes the structural hierarchy immediately visible.

6. **Entropy Distribution + Final μ Value**  
   Bar chart of island sizes (probabilities), plus full entropy calculation.  
   Shows exactly how μ emerges from structure, not style.

---

## 🧮 How μ Is Computed (Step-by-Step)

Given island-size probabilities \( p_i = \frac{s_i}{\sum s_i} \):

**Shannon Entropy:**  
\[
H = -\sum_i p_i \log(p_i)
\]

**Maximum entropy:**  
\[
H_{\max} = \log(K)
\]

**Cohesion:**  
\[
\mu = 1 - \frac{H}{H_{\max}}
\]

**Example from the notebook:**  
- \( H = 3.214 \)  
- \( H_{\max} = 5.554 \)  
\[
\mu = 1 - (3.214 / 5.554) = 0.421
\]

---

## 📊 Interpretation Guide (How to Read μ)

**HIGH μ (> 0.7)** — One dominant island  
- Unified structural composition  
- Strong portrait identity / clear geometric intent  

**MODERATE μ (0.4–0.7)** — Several strong islands  
- Balanced complexity  
- Common in Cézanne, Hopper, classical still-lifes  

**LOW μ (< 0.4)** — Many scattered islands  
- Structural fragmentation  
- Often seen in AI defaults, early iterations, noisy compositions  

This makes μ a **structural clarity index**, not a semantic measure.

---

## ❗ “This IS / This IS NOT” Clarification

### This **IS**  
✔ Gradient-field topology  
✔ Structural distribution  
✔ Entropy over meaningful gradient clusters  
✔ A perceptual proxy for visual coherence

### This is **NOT**  
✘ Object counting  
✘ Semantic segmentation  
✘ Noise-level estimation  

μ is computed **after** gradients are thresholded and clustered — ensuring you're reading **topological organization**, not pixel artifacts.

---

## 🧠 Why This Module Matters

This visualization lets reviewers literally watch μ being built:

- See the gradient forces  
- Watch the threshold mask filter signal  
- Watch 47 islands get extracted  
- See entropy computed in real numbers  
- See μ emerge from that math  

It eliminates the “magic number” problem.  
Cohesion isn’t a vibe — it’s the structure of the gradient field made measurable.

Run it on a portrait, a landscape, a chaotic AI output — you’ll immediately see how structural coherence behaves.

In [ ]:
# ============================================================
# VISUAL COMPANION: Gradient Field → Operation → Metric
# Example: μ (Cohesion) - From pixels to perceptual measure
# ============================================================

!pip install -q scikit-image opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from skimage import exposure
from skimage.measure import label, regionprops
from google.colab import files

# ---- Upload image ----
print("Upload an image to visualize cohesion (μ) computation...")
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using:", IMAGE_PATH)

# ============================================================
# STEP-BY-STEP COMPUTATION WITH VISUALIZATION
# ============================================================

img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError(f"Could not load: {IMAGE_PATH}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_f = gray.astype("float32") / 255.0

H, W = gray.shape

print(f"Image size: {W}×{H}")
print("\nComputing cohesion (μ) step-by-step...\n")

# ============================================================
# STEP 1: COMPUTE GRADIENT MAGNITUDE FIELD
# ============================================================
print("STEP 1: Compute gradient magnitude field G = |∇I|")

gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
grad_mag = np.sqrt(gx**2 + gy**2)
grad_mag_norm = exposure.rescale_intensity(grad_mag, out_range=(0, 1))

print(f"  → Gradient field computed: {grad_mag_norm.shape}")
print(f"  → Range: [{grad_mag_norm.min():.3f}, {grad_mag_norm.max():.3f}]")

# ============================================================
# STEP 2: THRESHOLD TO IDENTIFY HIGH-GRADIENT REGIONS
# ============================================================
print("\nSTEP 2: Threshold gradient → identify high-gradient regions")

threshold = np.percentile(grad_mag_norm, 75)
high_grad_mask = grad_mag_norm > threshold

print(f"  → Threshold: {threshold:.3f} (75th percentile)")
print(f"  → High-gradient pixels: {high_grad_mask.sum():,} / {high_grad_mask.size:,}")
print(f"  → Coverage: {high_grad_mask.sum() / high_grad_mask.size:.1%}")

# ============================================================
# STEP 3: LABEL CONNECTED COMPONENTS (ISLANDS)
# ============================================================
print("\nSTEP 3: Label connected components → gradient 'islands'")

labeled = label(high_grad_mask)
num_islands = labeled.max()

print(f"  → Number of islands (K): {num_islands}")

# Get island properties
props = regionprops(labeled)
areas = np.array([p.area for p in props]) if len(props) else np.array([1.0])

print(f"  → Island sizes: min={areas.min()}, max={areas.max()}, mean={areas.mean():.1f}")

# ============================================================
# STEP 4: COMPUTE ENTROPY OF ISLAND SIZES
# ============================================================
print("\nSTEP 4: Compute entropy of island size distribution")

# Normalize to probability distribution
p = areas / areas.sum()

# Shannon entropy
H = -np.sum(p * np.log2(p + 1e-12))
H_max = np.log2(len(p))

print(f"  → Probability distribution p: {len(p)} islands")
print(f"  → Shannon entropy H: {H:.3f}")
print(f"  → Maximum entropy H_max: {H_max:.3f}")
print(f"  → Normalized entropy H/H_max: {H/H_max:.3f}")

# ============================================================
# STEP 5: COMPUTE COHESION μ = 1 - (H / log K)
# ============================================================
print("\nSTEP 5: Compute cohesion μ = 1 - (H / H_max)")

mu = 1.0 - (H / H_max) if H_max > 0 else 0.0

print(f"  → Cohesion μ: {mu:.3f}")
print()
print("  Interpretation:")
if mu > 0.7:
    print(f"    HIGH cohesion ({mu:.3f}) → One dominant structure")
    print("    → Unified composition, single visual mass")
elif mu > 0.4:
    print(f"    MODERATE cohesion ({mu:.3f}) → Several major structures")
    print("    → Balanced composition with multiple centers")
else:
    print(f"    LOW cohesion ({mu:.3f}) → Many scattered fragments")
    print("    → Fragmented composition, dispersed energy")

# ============================================================
# STEP 6: VISUALIZE THE FIVE LARGEST ISLANDS
# ============================================================
print("\nSTEP 6: Color-code the five largest islands")

# Sort by area
sorted_props = sorted(props, key=lambda p: p.area, reverse=True)
top_n = min(5, len(sorted_props))

# Create colored island map
island_viz = np.zeros((*labeled.shape, 3), dtype=np.uint8)
colors = [
    [255, 0, 0],      # Red - largest
    [0, 255, 0],      # Green - 2nd
    [0, 0, 255],      # Blue - 3rd
    [255, 255, 0],    # Yellow - 4th
    [255, 0, 255]     # Magenta - 5th
]

for i in range(top_n):
    mask = labeled == sorted_props[i].label
    island_viz[mask] = colors[i]

print(f"  → Top {top_n} islands colored and visualized")
print(f"  → Largest island: {sorted_props[0].area:,} pixels ({sorted_props[0].area/high_grad_mask.sum():.1%} of high-gradient mass)")

# ============================================================
# CREATE 3x2 VISUALIZATION GRID
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(16, 20))

# 1. Original Image
axes[0, 0].imshow(img)
axes[0, 0].set_title("Step 0: Original Image", fontsize=13, weight='bold')
axes[0, 0].axis('off')

# 2. Gradient Magnitude Field
axes[0, 1].imshow(grad_mag_norm, cmap='hot')
axes[0, 1].set_title(f"Step 1: Gradient Field |∇I|\nRange: [{grad_mag_norm.min():.3f}, {grad_mag_norm.max():.3f}]",
                     fontsize=13, weight='bold')
axes[0, 1].axis('off')

# 3. Thresholded High-Gradient Mask
axes[1, 0].imshow(high_grad_mask, cmap='gray')
axes[1, 0].set_title(f"Step 2: High-Gradient Mask\nThreshold: {threshold:.3f} (75th percentile)\n{high_grad_mask.sum():,} pixels",
                     fontsize=13, weight='bold')
axes[1, 0].axis('off')

# 4. Labeled Islands (color-coded by ID)
# Use a colormap to show distinct islands
island_display = labeled.copy().astype(float)
island_display[island_display == 0] = np.nan  # Make background transparent
axes[1, 1].imshow(img, alpha=0.3)
axes[1, 1].imshow(island_display, cmap='tab20', alpha=0.7)
axes[1, 1].set_title(f"Step 3: Connected Components\nK = {num_islands} islands",
                     fontsize=13, weight='bold')
axes[1, 1].axis('off')

# 5. Top 5 Islands Highlighted
axes[2, 0].imshow(img, alpha=0.4)
axes[2, 0].imshow(island_viz, alpha=0.6)
axes[2, 0].set_title(f"Step 6: Top 5 Largest Islands\nLargest: {sorted_props[0].area:,} px ({sorted_props[0].area/high_grad_mask.sum():.1%})",
                     fontsize=13, weight='bold')
axes[2, 0].axis('off')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=np.array(colors[i])/255, label=f"#{i+1} ({sorted_props[i].area:,} px)")
    for i in range(min(top_n, len(sorted_props)))
]
axes[2, 0].legend(handles=legend_elements, loc='upper right', fontsize=9)

# 6. Entropy Visualization (bar chart of island sizes)
axes[2, 1].bar(range(len(areas)), sorted(areas, reverse=True), color='#FF6B6B', edgecolor='black')
axes[2, 1].set_yscale('log')
axes[2, 1].set_xlabel('Island Rank', fontsize=11, weight='bold')
axes[2, 1].set_ylabel('Island Size (pixels)', fontsize=11, weight='bold')
axes[2, 1].set_title(f"Step 4: Island Size Distribution\nEntropy H = {H:.3f}, H_max = {H_max:.3f}\nμ = 1 - (H/H_max) = {mu:.3f}",
                     fontsize=13, weight='bold')
axes[2, 1].grid(axis='y', alpha=0.3)

# Add cohesion interpretation box
textbox = f"""
COHESION μ = {mu:.3f}

{"HIGH" if mu > 0.7 else "MODERATE" if mu > 0.4 else "LOW"} COHESION

• {num_islands} gradient islands
• Largest island: {sorted_props[0].area/high_grad_mask.sum():.1%} of mass
• Entropy: {H:.3f} / {H_max:.3f}

{"→ One dominant structure" if mu > 0.7 else "→ Multiple structures" if mu > 0.4 else "→ Fragmented field"}
{"→ Unified composition" if mu > 0.7 else "→ Balanced complexity" if mu > 0.4 else "→ Dispersed energy"}
"""

axes[2, 1].text(0.98, 0.97, textbox, transform=axes[2, 1].transAxes,
               fontsize=10, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle("From Gradient Field to Cohesion Metric (μ): Complete Pipeline",
             fontsize=16, weight='bold', y=0.995)
plt.tight_layout()
plt.show()

# ============================================================
# PRINT MATHEMATICAL FORMULA
# ============================================================
print("\n" + "="*70)
print("MATHEMATICAL FORMULA:")
print("="*70)
print()
print("μ (Cohesion) = 1 - (H / H_max)")
print()
print("Where:")
print(f"  H = Shannon entropy = -∑ pᵢ log₂(pᵢ)")
print(f"    = {H:.3f}")
print()
print(f"  H_max = log₂(K) where K = number of islands")
print(f"        = log₂({num_islands})")
print(f"        = {H_max:.3f}")
print()
print(f"  pᵢ = areaᵢ / ∑ areas (probability distribution)")
print()
print("Result:")
print(f"  μ = 1 - ({H:.3f} / {H_max:.3f})")
print(f"    = {mu:.3f}")
print()
print("="*70)
print("INTERPRETATION:")
print("="*70)
print()
print("HIGH μ (→ 1.0):")
print("  • One island dominates")
print("  • Low entropy (predictable)")
print("  • Unified visual mass")
print("  • Example: Single portrait subject")
print()
print("LOW μ (→ 0.0):")
print("  • Many islands of similar size")
print("  • High entropy (uniform distribution)")
print("  • Fragmented composition")
print("  • Example: Scattered objects, busy scene")
print()
print("This is NOT:")
print("  ✗ Object counting")
print("  ✗ Semantic segmentation")
print("  ✗ Subject detection")
print()
print("This IS:")
print("  ✓ Gradient field topology")
print("  ✓ Perceptual mass distribution")
print("  ✓ Compositional unity measure")
print()
print("Cohesion measures: Does the frame read as ONE thing or MANY things?")
print("It's a pure geometric property of the gradient field.")
print("="*70)
